In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import os
from copy import deepcopy

# ─── Import des modules NRC ──────────────────────────────────────────────────
from pipeline import run_nrc_pipeline


from utils import setup_logging
log_path=setup_logging("logs/run_YYYYMMDD_HHMMSS.log")


import logging
setup_logging(level=logging.DEBUG)





11:34:47 [INFO    ] utils — Logging initialisé → logs/run_DDMMYYYY_HHMMSS.log\run_20260428_113447.log
11:34:47 [INFO    ] utils — Logging initialisé → logs\run_20260428_113447.log


'logs\\run_20260428_113447.log'

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

batch_size = 64

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=transform)
testset  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True,  num_workers=2)
testloader  = torch.utils.data.DataLoader(testset,  batch_size=batch_size, shuffle=False, num_workers=2)

classes = ('plane','car','bird','cat','deer','dog','frog','horse','ship','truck')
print(f"Train: {len(trainset)} samples | Test: {len(testset)} samples")


In [ ]:
def imshow(img):
    img = img / 2 + 0.5
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

dataiter = iter(trainloader)
images, labels = next(dataiter)
imshow(torchvision.utils.make_grid(images[:4]))
print(' '.join(f'{classes[labels[j]]:5s}' for j in range(4)))


## Méthode NRC reformulée en 4 étapes

Ce notebook est organisé pour distinguer clairement les quatre blocs de la méthode proposée :

1. **InitialNet** : entraîner un réseau de référence sur CIFAR-10.
2. **PCA avec un seul paramètre `eTh`** : analyser chaque couche convolutionnelle et ne conserver que l'information expliquée par le seuil d'énergie. Pas de randomisation à cette étape.
3. **Pseudo-randomization avec LFSR + distance de Grassmann (`p`)** : utiliser le ratio `p` pour décider combien de vecteurs PCA sont remplacés par des vecteurs LFSR sélectionnés via la distance de Grassmann.
4. **Ré-entraînement partiel** : geler `conv_base`, n'entraîner que `conv_coord` et éventuellement les couches fully-connected.

### Fil conducteur

```
Étape 1 → apprendre InitialNet
Étape 2 → appliquer la PCA sur les poids de InitialNet avec le seul argument eTh
Étape 3 → pseudo-randomiser la base via LFSR + Grassmann avec l'argument p
Étape 4 → reconstruire un bloc NRC puis ré-entraîner partiellement
```


### Utilitaires partagés — LFSR (générateur pseudo-aléatoire) & Distance de Grassmann

In [ ]:
from utils import _grassmann_distance, _prng_vector
# Utilitaires importés depuis utils.py :
#   _grassmann_distance(A, B) → float
#   _prng_vector(seed, dim)   → np.ndarray
#
# Voir utils.py pour le détail de l'implémentation LFSR + Box-Muller.
print("✓ utils.py chargé : _grassmann_distance, _prng_vector")


### Étape 1 — InitialNet : modèle de référence

On commence par entraîner un CNN de base sur CIFAR-10. Ce modèle sert de **point de départ** à toute la méthode.

Architecture utilisée : `conv1(3→n_filters, 5×5) → pool → conv2(n_filters→16, 5×5) → pool → fc1 → fc2 → fc3`.

À la fin de cette étape, on dispose des poids du réseau initial, qui seront ensuite analysés et transformés par la méthode NRC.

In [ ]:
from step1_initial_net import InitialNet
# Réseau initial importé depuis step1_initial_net.py :
#   InitialNet(n_filters=6) — LeNet-like CNN (conv1→conv2→fc1→fc2→fc3)
#
# Voir step1_initial_net.py pour l'architecture et save_weights/load_weights.
print("✓ step1_initial_net.py chargé : InitialNet")


### Étape 2 — PCA avec un seul paramètre : `eTh`

Cette étape applique une **ACP sur les filtres convolutionnels** du réseau initial.

Le seul paramètre de contrôle est `eTh` (energy threshold). Aucun ratio de randomisation n'intervient ici.

- `eTh` fixe la fraction minimale d'énergie cumulée à conserver.
- On garde les `s` premières composantes principales telles que la variance cumulée atteigne ce seuil.
- On obtient une base réduite `B_T` (s × d) et les coordonnées associées `Y_T`.

Autrement dit, cette étape répond à la question : **quelle partie de l'information des filtres suffit à représenter la couche ?**

Le paramètre `p` (randomization_ratio) n'apparaît qu'à l'étape 3.


In [ ]:
from step2_pca import _analyze_conv_nrc, FilterAnalyzer
# Analyse PCA importée depuis step2_pca.py :
#   _analyze_conv_nrc(conv, energy_threshold, name) → dict
#   FilterAnalyzer(energy_threshold)  .analyze(model) → self
#
# Paramètre unique : eTh (seuil d'énergie PCA).
# Voir step2_pca.py pour le détail.
print("✓ step2_pca.py chargé : _analyze_conv_nrc, FilterAnalyzer")


### Étape 3 — Pseudo-randomization avec LFSR et distance de Grassmann (`p`)

Le paramètre `p` (randomization_ratio) contrôle cette étape.

À partir de la base PCA `B_T` de taille `s` obtenue à l'étape 2 :

- `n_keep = max(1, round(s × (1 − p)))` vecteurs PCA sont **conservés** (Vs),
- `n_rand = s − n_keep` vecteurs sont **remplacés** par des vecteurs LFSR (Vr).

La sélection des vecteurs LFSR ne se fait pas au hasard pur :

- on génère `K` candidats via LFSR pour chaque direction à remplacer,
- on mesure leur qualité avec la **distance de Grassmann**,
- on retient les graines qui rapprochent le mieux la base hybride du sous-espace cible.

Résultat : une base hybride `BNRC` de rang `t = s`, où une partie est stockée et l'autre est régénérable à partir de simples graines.


In [ ]:
from step3_pseudo_random import _build_bnrc, PseudoRandomFilterSelector
# Pseudo-randomisation importée depuis step3_pseudo_random.py :
#   _build_bnrc(analysis, randomization_ratio, K, seed, name) → dict
#   PseudoRandomFilterSelector(randomization_ratio, K, seed)  .select(analyzer) → self
#
# Paramètre unique : p (ratio de pseudo-randomisation).
# Voir step3_pseudo_random.py pour LFSR + Grassmann.
print("✓ step3_pseudo_random.py chargé : _build_bnrc, PseudoRandomFilterSelector")


### Étape 4 — Ré-entraînement partiel : construction du bloc NRC

Pour pouvoir ré-entraîner le modèle après pseudo-randomization, chaque couche convolutionnelle originale est remplacée par un **bloc NRC** composé de deux sous-couches :

| Sous-couche | Rôle | Statut |
|-------------|------|--------|
| `conv_base` | projection dans la base hybride `BNRC` | gelée |
| `conv_coord` | recombinaison linéaire des directions | entraînable |

Le bloc sépare donc clairement :

- la partie **structurelle** du filtre, fixée par `BNRC`,
- la partie **adaptative**, portée par les coordonnées `Y_NRC`.

Cette factorisation prépare directement l'étape de ré-entraînement partiel.

In [ ]:
from step4_nrc_net import NRCConvBlock, NRCNet, train_one_epoch, evaluate
# Architecture NRC importée depuis step4_nrc_net.py :
#   NRCConvBlock(bnrc_data, Y_nrc, ...)  — conv_base (gelé) + conv_coord (entraînable)
#   NRCNet(model, analyzer, selector)    — réseau complet avec blocs NRC
#
# Voir step4_nrc_net.py pour l'architecture.
print("✓ step4_nrc_net.py chargé : NRCConvBlock, NRCNet")


### Étape 4 — Ré-entraînement partiel des coordonnées

Dans cette dernière étape, on ne ré-entraîne pas tout le réseau de zéro.

Le principe est le suivant :

- `conv_base` reste **gelé**, car il représente la base hybride fixée après PCA + pseudo-randomization ;
- `conv_coord` est **ré-entraîné**, car il porte les coefficients de recombinaison ;
- les couches fully-connected peuvent être laissées entraînables ou figées selon l'expérience visée.

L'objectif est d'adapter finement le modèle compressé sans perdre le bénéfice structurel obtenu aux étapes 2 et 3.

In [ ]:
# Fonctions d'entraînement importées depuis step4_nrc_net.py :
#   train_one_epoch(model, loader, criterion, optimizer, device) → loss
#   evaluate(model, loader, device) → accuracy (%)
#
# Voir step4_nrc_net.py pour les boucles d'entraînement.
print("✓ step4_nrc_net.py chargé : train_one_epoch, evaluate")


### Annexe — Sauvegarde compressée et restauration

Cette partie ne constitue pas une étape supplémentaire de la méthode.

Elle sert à sérialiser le modèle obtenu après les 4 étapes précédentes, puis à vérifier qu'il peut être restauré correctement.

Le principe reste le même :

- `conv_base` n'est pas stocké explicitement ;
- `conv_coord` et les couches fully-connected sont stockés ;
- les informations nécessaires à la reconstruction sont conservées sous forme compacte.

In [ ]:
from compressed_net import CompressedNet
# Sauvegarde compressée importée depuis compressed_net.py :
#   CompressedNet.save(nrc_net, analyzer, selector, path_json, path_pth)
#   CompressedNet.load(path_json, path_pth, n_filters) → NRCNet
#
# Voir compressed_net.py pour la sérialisation.
print("✓ compressed_net.py chargé : CompressedNet")


### Expérience ciblée — Étapes 1 et 2 uniquement

Cette section isole le comportement de la méthode jusqu'à la **PCA seule**.

But : comparer `InitialNet` avec une version factorisée issue uniquement de l'étape 2, sans pseudo-randomization, afin d'observer l'effet du paramètre `eTh` indépendamment des étapes suivantes.

In [ ]:
# Pipeline partiel : entraînement 2 epochs + suivi des performances avec PCA sur InitialNet
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    criterion = nn.CrossEntropyLoss()
    initial_net = InitialNet().to(device)
    optimizer = optim.Adam(initial_net.parameters(), lr=1e-3)
    print("[Main] Modèle initial créé.")

    energy_threshold = 0.75  # conserver tous les vecteurs PCA (pas de réduction) pour l'analyse comparative

    for epoch in range(1, 3):
        train_loss, train_acc = train_one_epoch(
            initial_net, trainloader, optimizer, criterion, device
        )
        val_loss, val_acc = evaluate(initial_net, testloader, criterion, device)

        print(f"\nEpoch {epoch}/2")
        print(f"  InitialNet train | loss={train_loss:.4f} | acc={train_acc:.2%}")
        print(f"  InitialNet test  | loss={val_loss:.4f} | acc={val_acc:.2%}")

        base_for_pca = deepcopy(initial_net).cpu()

        # Étape 2 : PCA seule (eTh)
        analyzer_pca = FilterAnalyzer(energy_threshold=energy_threshold)
        analyzer_pca.analyze(base_for_pca.conv1, base_for_pca.conv2)

        # Étape 3 : p=0 → aucun remplacement LFSR (PCA pure)
        selector_pca = PseudoRandomFilterSelector(
            randomization_ratio=0.0, K=0, seed=0,
        )
        selector_pca.select(analyzer_pca)

        pca_net = NRCNet(
            initial_net=base_for_pca,
            analyzer=analyzer_pca,
            selector=selector_pca,
            freeze_fc=False,
        ).to(device)

        # PCA seule : pas d'entraînement, juste évaluation pour voir l'impact de la réduction PCA sans randomisation
        pca_train_loss, pca_train_acc = evaluate(pca_net, trainloader, criterion, device)
        pca_test_loss, pca_test_acc = evaluate(pca_net, testloader, criterion, device)

        print(f"  PCA train        | loss={pca_train_loss:.4f} | acc={pca_train_acc:.2%}")
        print(f"  PCA test         | loss={pca_test_loss:.4f} | acc={pca_test_acc:.2%}")
        print(f"  Delta test acc   | {pca_test_acc - val_acc:+.2%}")

        print("  PCA Analysis:")
        for conv_name, conv_data in [("conv1", analyzer_pca.conv1), ("conv2", analyzer_pca.conv2)]:
            b = selector_pca.conv1_bnrc if conv_name == "conv1" else selector_pca.conv2_bnrc
            s = conv_data["s"]
            energy = conv_data["energy_captured"]
            n_keep = b["n_keep"]
            n_rand = b["n_rand"]
            print(
                f"    {conv_name}: s={s} | énergie capturée={energy:.1%} "
                f"| Vs={n_keep} | Vr={n_rand}"
            )

    print("\n[Test terminé] 2 epochs exécutées avec comparaison InitialNet vs PCA.")


In [ ]:
# ─── Impact PCA seule (sans ré-entraînement) pour différents eTh ──────────────

criterion = nn.CrossEntropyLoss()

# Entraîner un InitialNet frais sur 2 epochs
net_ref = InitialNet().to(device)
opt_ref = optim.Adam(net_ref.parameters(), lr=1e-3)
for ep in range(2):
    train_one_epoch(net_ref, trainloader, opt_ref, criterion, device)
ref_loss, ref_acc = evaluate(net_ref, testloader, criterion, device)
ref_params = sum(p.numel() for p in net_ref.parameters())
ref_bytes = ref_params * 4

print(f"InitialNet (2 epochs) : acc={ref_acc:.2%} | params={ref_params:,}\n")

eTh_list = [0.25, 0.50, 0.75, 0.90]
table = []

for eTh in eTh_list:
    base = deepcopy(net_ref).cpu()
    anlz = FilterAnalyzer(energy_threshold=eTh)
    anlz.analyze(base.conv1, base.conv2)

    sel = PseudoRandomFilterSelector(randomization_ratio=0.0, K=0, seed=0)
    sel.select(anlz)

    pca_m = NRCNet(initial_net=base, analyzer=anlz, selector=sel, freeze_fc=False).to(device)

    # Évaluation sans ré-entraînement
    test_loss, test_acc = evaluate(pca_m, testloader, criterion, device)
    n_params = sum(p.numel() for p in pca_m.parameters())
    mem_bytes = n_params * 4
    gain = (1 - n_params / ref_params) * 100

    table.append({
        "eTh": eTh, "acc": test_acc, "params": n_params,
        "bytes": mem_bytes, "gain": gain,
        "s1": anlz.conv1["s"], "s2": anlz.conv2["s"],
    })

# ─── Tableau ──────────────────────────────────────────────────────────────────
print(f"{'eTh':>6} | {'Accuracy':>10} | {'Params':>10} | {'Mémoire':>10} | {'Gain':>8} | {'s_conv1':>7} | {'s_conv2':>7}")
print("─" * 80)
for r in table:
    mem_str = f"{r['bytes']/1024:.1f} KB"
    print(f"{r['eTh']:>5.0%} | {r['acc']:>9.2%} | {r['params']:>10,} | {mem_str:>10} | {r['gain']:>+7.1f}% | {r['s1']:>7} | {r['s2']:>7}")
print(f"{'Ref':>6} | {ref_acc:>9.2%} | {ref_params:>10,} | {ref_bytes/1024:.1f} KB | {'—':>8} | {'—':>7} | {'—':>7}")

# ─── Visualisation ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(10, 15))
fig.suptitle("Impact de la PCA seule (sans ré-entraînement, 2 epochs)", fontsize=14, fontweight="bold")

x_labels = [f"{r['eTh']:.0%}" for r in table]
accs = [r["acc"] * 100 for r in table]
mems = [r["bytes"] / 1024 for r in table]
n_params_list = [r["params"] for r in table]
gains_pct = [r["gain"] for r in table]

# Graphe 1 : Accuracy
bars1 = axes[0].bar(x_labels, accs, color="#4C72B0", edgecolor="black", linewidth=0.5)
axes[0].axhline(ref_acc * 100, color="red", linestyle="--", linewidth=1.5, label=f"Initial ({ref_acc:.2%})")
for bar, val in zip(bars1, accs):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 f"{val:.1f}%", ha="center", fontsize=10, fontweight="bold")
axes[0].set_xlabel("Seuil d'énergie (eTh)")
axes[0].set_ylabel("Test Accuracy (%)")
axes[0].set_title("Précision après PCA seule")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

# Graphe 2 : Taille mémoire + nombre de paramètres en dessous
bars2 = axes[1].bar(x_labels, mems, color="#55A868", edgecolor="black", linewidth=0.5)
axes[1].axhline(ref_bytes / 1024, color="red", linestyle="--", linewidth=1.5, label=f"Initial ({ref_bytes/1024:.1f} KB)")
for bar, val, n_p in zip(bars2, mems, n_params_list):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                 f"{val:.1f} KB\n({n_p:,} params)", ha="center", fontsize=9, fontweight="bold")
axes[1].set_xlabel("Seuil d'énergie (eTh)")
axes[1].set_ylabel("Taille mémoire (KB)")
axes[1].set_title("Taille mémoire du modèle PCA")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

# Graphe 3 : Gain mémoire
bars3 = axes[2].bar(x_labels, gains_pct, color="#C44E52", edgecolor="black", linewidth=0.5)
for bar, val in zip(bars3, gains_pct):
    axes[2].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
                 f"{val:.1f}%", ha="center", fontsize=10, fontweight="bold")
axes[2].set_xlabel("Seuil d'énergie (eTh)")
axes[2].set_ylabel("Gain de compression (%)")
axes[2].set_title("Réduction des paramètres vs Initial")
axes[2].axhline(0, color="gray", linestyle="-", linewidth=0.5)
axes[2].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# ─── Impact PCA seule (sans ré-entraînement) pour différents eTh ──────────────

criterion = nn.CrossEntropyLoss()

# ── InitialNet de référence (2 epochs) ────────────────────────────────────────
net_ref = InitialNet().to(device)
opt_ref = optim.Adam(net_ref.parameters(), lr=1e-3)
for ep in range(2):
    train_one_epoch(net_ref, trainloader, opt_ref, criterion, device)
ref_loss, ref_acc = evaluate(net_ref, testloader, criterion, device)

# Paramètres conv de référence — cohérent avec le calcul NRC ci-dessous
ref_conv_params = (net_ref.conv1.weight.numel() + net_ref.conv1.bias.numel() +
                   net_ref.conv2.weight.numel() + net_ref.conv2.bias.numel())
ref_conv_bytes  = ref_conv_params * 4

print(f"InitialNet (2 epochs) : acc={ref_acc:.2%} | conv params={ref_conv_params:,}\n")

eTh_list = [0.25, 0.50, 0.75, 0.90]
table = []

for eTh in eTh_list:
    base = deepcopy(net_ref).cpu()

    # ── Étape 2 : PCA ─────────────────────────────────────────────────────────
    anlz = FilterAnalyzer(energy_threshold=eTh)
    anlz.analyze(base.conv1, base.conv2)

    # ── Étape 3 : PCA pure, pas de LFSR ───────────────────────────────────────
    sel = PseudoRandomFilterSelector(randomization_ratio=0.0, K=0, seed=0)
    sel.select(anlz)

    # ── Construction NRCNet + inférence directe (pas de fine-tuning) ──────────
    pca_m = NRCNet(initial_net=base, analyzer=anlz, selector=sel,
                   freeze_fc=False).to(device)
    _, test_acc = evaluate(pca_m, testloader, criterion, device)

    # Paramètres conv NRC — même périmètre que ref_conv_params
    nrc_conv_params = (sum(p.numel() for p in pca_m.nrc_conv1.parameters()) +
                       sum(p.numel() for p in pca_m.nrc_conv2.parameters()))
    conv_bytes = nrc_conv_params * 4
    conv_gain  = (1 - nrc_conv_params / ref_conv_params) * 100

    table.append({
        "eTh": eTh, "acc": test_acc,
        "conv_params": nrc_conv_params,
        "conv_bytes":  conv_bytes,
        "conv_gain":   conv_gain,
        "s1": anlz.conv1["s"], "s2": anlz.conv2["s"],
    })

# ─── Tableau ──────────────────────────────────────────────────────────────────
print(f"{'eTh':>6} | {'Accuracy':>10} | {'Conv params':>12} | {'Mémoire conv':>13} | {'Gain conv':>10} | {'s₁':>4} | {'s₂':>4}")
print("─" * 82)
for r in table:
    print(f"{r['eTh']:>5.0%} | {r['acc']:>9.2%} | {r['conv_params']:>12,} | "
          f"{r['conv_bytes']/1024:>11.1f} KB | {r['conv_gain']:>+9.1f}% | "
          f"{r['s1']:>4} | {r['s2']:>4}")
print(f"{'Réf':>6} | {ref_acc:>9.2%} | {ref_conv_params:>12,} | "
      f"{ref_conv_bytes/1024:>11.1f} KB | {'—':>10} | {'—':>4} | {'—':>4}")

# ─── Visualisation ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(10, 15))
fig.suptitle("Impact de la PCA seule (sans ré-entraînement, 2 epochs)",
             fontsize=14, fontweight="bold")

x_labels   = [f"{r['eTh']:.0%}"        for r in table]
accs       = [r["acc"] * 100           for r in table]
conv_mems  = [r["conv_bytes"] / 1024   for r in table]
conv_gains = [r["conv_gain"]           for r in table]

# Graphe 1 : Accuracy
bars1 = axes[0].bar(x_labels, accs, color="#4C72B0", edgecolor="black", linewidth=0.5)
axes[0].axhline(ref_acc * 100, color="red", linestyle="--", linewidth=1.5,
                label=f"Initial ({ref_acc:.2%})")
for bar, val in zip(bars1, accs):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 f"{val:.1f}%", ha="center", fontsize=10, fontweight="bold")
axes[0].set_xlabel("Seuil d'énergie (eTh)")
axes[0].set_ylabel("Test Accuracy (%)")
axes[0].set_title("Précision après PCA seule")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

# Graphe 2 : Taille mémoire conv
bars2 = axes[1].bar(x_labels, conv_mems, color="#55A868", edgecolor="black", linewidth=0.5)
axes[1].axhline(ref_conv_bytes / 1024, color="red", linestyle="--", linewidth=1.5,
                label=f"Conv initial ({ref_conv_bytes/1024:.1f} KB)")
for bar, val, r in zip(bars2, conv_mems, table):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                 f"{val:.2f} KB\n({r['conv_params']:,} params)\ns₁={r['s1']}, s₂={r['s2']}",
                 ha="center", fontsize=8, fontweight="bold")
axes[1].set_xlabel("Seuil d'énergie (eTh)")
axes[1].set_ylabel("Mémoire couches conv (KB)")
axes[1].set_title("Taille mémoire — couches de convolution uniquement")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

# Graphe 3 : Gain de compression conv
bars3 = axes[2].bar(x_labels, conv_gains, color="#C44E52", edgecolor="black", linewidth=0.5)
for bar, val in zip(bars3, conv_gains):
    axes[2].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 f"{val:.1f}%", ha="center", fontsize=10, fontweight="bold")
axes[2].set_xlabel("Seuil d'énergie (eTh)")
axes[2].set_ylabel("Gain de compression (%)")
axes[2].set_title("Réduction des paramètres conv vs conv initial")
axes[2].axhline(0, color="gray", linestyle="-", linewidth=0.5)
axes[2].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Export Excel : ACP sans ré-entraînement ─────────────────────────────────
import pandas as pd

_rows = []
for r in table:
    _rows.append({
        "eTh": f"{r['eTh']:.0%}",
        "Accuracy (%)": round(r["acc"] * 100, 2),
        "Conv params": r["conv_params"],
        "Mémoire conv (KB)": round(r["conv_bytes"] / 1024, 2),
        "Gain compression (%)": round(r["conv_gain"], 2),
        "s_conv1": r["s1"],
        "s_conv2": r["s2"],
        "Ré-entraînement": "Non",
    })
_rows.append({
    "eTh": "Référence",
    "Accuracy (%)": round(ref_acc * 100, 2),
    "Conv params": ref_conv_params,
    "Mémoire conv (KB)": round(ref_conv_bytes / 1024, 2),
    "Gain compression (%)": None,
    "s_conv1": "—",
    "s_conv2": "—",
    "Ré-entraînement": "—",
})
_df = pd.DataFrame(_rows)
_out = "ACP_sans_reentrainement.xlsx"
_df.to_excel(_out, index=False, sheet_name="PCA_sans_FT")
print(f"[Export] {_out} généré ({len(_rows) - 1} configurations + ligne référence).")


## PCA seule avec ré-entrainement 

In [ ]:
criterion = nn.CrossEntropyLoss()

# ── Référence : InitialNet entraîné 2 epochs ───────────────────────────────────
net_ref = InitialNet().to(device)
opt_ref = optim.Adam(net_ref.parameters(), lr=1e-3)
for ep in range(2):
    train_one_epoch(net_ref, trainloader, opt_ref, criterion, device)
_, ref_acc = evaluate(net_ref, testloader, criterion, device)
ref_conv_params = (net_ref.conv1.weight.numel() + net_ref.conv1.bias.numel() +
                   net_ref.conv2.weight.numel() + net_ref.conv2.bias.numel())
ref_conv_bytes  = ref_conv_params * 4   # ← ajout pour cohérence avec visualisation
print(f"InitialNet (2 epochs) : acc={ref_acc:.2%} | conv params={ref_conv_params:,}\n")

# ── PCA + fine-tuning : budget total = 2 epochs, depuis zéro ──────────────────
eTh_list = [0.25, 0.50, 0.75, 0.90]
table_ft = []

for eTh in eTh_list:
    print(f"\n{'─'*60}")
    print(f"  Run indépendant — eTh = {eTh:.0%}")
    print(f"{'─'*60}")

    # InitialNet VIERGE — aucun entraînement préalable
    base_raw = InitialNet().cpu()

    # Étape 2 : PCA
    anlz = FilterAnalyzer(energy_threshold=eTh)
    anlz.analyze(base_raw.conv1, base_raw.conv2)

    # Étape 3 : PCA pure, pas de LFSR
    sel = PseudoRandomFilterSelector(randomization_ratio=0.0, K=0, seed=0)
    sel.select(anlz)

    # Étape 4 : NRCNet + 2 epochs
    nrc = NRCNet(initial_net=base_raw, analyzer=anlz, selector=sel,
                 freeze_fc=False).to(device)
    trainable = [p for p in nrc.parameters() if p.requires_grad]
    opt_ft = optim.Adam(trainable, lr=1e-3)

    for ep in range(1, 3):
        tl, ta = train_one_epoch(nrc, trainloader, opt_ft, criterion, device)
        vl, va = evaluate(nrc, testloader, criterion, device)
        print(f"  Ép {ep}/2 | train acc={ta:.2%} | val acc={va:.2%}")

    acc_ft = va

    # ── Métriques — mêmes clés que le code sans ré-entraînement ──────────────
    conv_params = (sum(p.numel() for p in nrc.nrc_conv1.parameters()) +
                   sum(p.numel() for p in nrc.nrc_conv2.parameters()))
    conv_bytes  = conv_params * 4                          # ← même clé
    conv_gain   = (1 - conv_params / ref_conv_params) * 100  # ← même clé

    table_ft.append({
        "eTh"        : eTh,
        "acc"        : acc_ft,
        "conv_params": conv_params,
        "conv_bytes" : conv_bytes,   # ← aligné avec code sans FT
        "conv_gain"  : conv_gain,    # ← aligné avec code sans FT
        "s1"         : anlz.conv1["s"],
        "s2"         : anlz.conv2["s"],
    })

# ── Tableau ────────────────────────────────────────────────────────────────────
print(f"\n{'eTh':>6} | {'Accuracy':>10} | {'Conv params':>12} | {'Mémoire conv':>13} | {'Gain conv':>10} | {'s₁':>4} | {'s₂':>4}")
print("─" * 82)
for r in table_ft:
    print(f"{r['eTh']:>5.0%} | {r['acc']:>9.2%} | {r['conv_params']:>12,} | "
          f"{r['conv_bytes']/1024:>11.1f} KB | {r['conv_gain']:>+9.1f}% | "
          f"{r['s1']:>4} | {r['s2']:>4}")
print(f"{'Réf':>6} | {ref_acc:>9.2%} | {ref_conv_params:>12,} | "
      f"{ref_conv_bytes/1024:>11.1f} KB | {'—':>10} | {'—':>4} | {'—':>4}")

# ── Visualisation ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(10, 15))
fig.suptitle("Impact de la PCA seule (avec ré-entraînement partiel, 2 epochs)",
             fontsize=14, fontweight="bold")

x_labels   = [f"{r['eTh']:.0%}"       for r in table_ft]
accs       = [r["acc"] * 100          for r in table_ft]
conv_mems  = [r["conv_bytes"] / 1024  for r in table_ft]
conv_gains = [r["conv_gain"]          for r in table_ft]

# Graphe 1 : Accuracy
bars1 = axes[0].bar(x_labels, accs, color="#4C72B0", edgecolor="black", linewidth=0.5)
axes[0].axhline(ref_acc * 100, color="red", linestyle="--", linewidth=1.5,
                label=f"Initial ({ref_acc:.2%})")
for bar, val in zip(bars1, accs):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 f"{val:.1f}%", ha="center", fontsize=10, fontweight="bold")
axes[0].set_xlabel("Seuil d'énergie (eTh)")
axes[0].set_ylabel("Test Accuracy (%)")
axes[0].set_title("Précision après PCA + ré-entraînement partiel")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

# Graphe 2 : Taille mémoire conv
bars2 = axes[1].bar(x_labels, conv_mems, color="#55A868", edgecolor="black", linewidth=0.5)
axes[1].axhline(ref_conv_bytes / 1024, color="red", linestyle="--", linewidth=1.5,
                label=f"Conv initial ({ref_conv_bytes/1024:.1f} KB)")
for bar, val, r in zip(bars2, conv_mems, table_ft):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                 f"{val:.2f} KB\n({r['conv_params']:,} params)\ns₁={r['s1']}, s₂={r['s2']}",
                 ha="center", fontsize=8, fontweight="bold")
axes[1].set_xlabel("Seuil d'énergie (eTh)")
axes[1].set_ylabel("Mémoire couches conv (KB)")
axes[1].set_title("Taille mémoire — couches de convolution uniquement")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

# Graphe 3 : Gain de compression conv
bars3 = axes[2].bar(x_labels, conv_gains, color="#C44E52", edgecolor="black", linewidth=0.5)
for bar, val in zip(bars3, conv_gains):
    axes[2].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 f"{val:.1f}%", ha="center", fontsize=10, fontweight="bold")
axes[2].set_xlabel("Seuil d'énergie (eTh)")
axes[2].set_ylabel("Gain de compression (%)")
axes[2].set_title("Réduction des paramètres conv vs conv initial")
axes[2].axhline(0, color="gray", linestyle="-", linewidth=0.5)
axes[2].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Export Excel : ACP avec ré-entraînement ──────────────────────────────────
import pandas as pd

_rows = []
for r in table_ft:
    _rows.append({
        "eTh": f"{r['eTh']:.0%}",
        "Accuracy (%)": round(r["acc"] * 100, 2),
        "Conv params": r["conv_params"],
        "Mémoire conv (KB)": round(r["conv_bytes"] / 1024, 2),
        "Gain compression (%)": round(r["conv_gain"], 2),
        "s_conv1": r["s1"],
        "s_conv2": r["s2"],
        "Ré-entraînement": "Oui",
    })
_rows.append({
    "eTh": "Référence",
    "Accuracy (%)": round(ref_acc * 100, 2),
    "Conv params": ref_conv_params,
    "Mémoire conv (KB)": round(ref_conv_bytes / 1024, 2),
    "Gain compression (%)": None,
    "s_conv1": "—",
    "s_conv2": "—",
    "Ré-entraînement": "—",
})
_df = pd.DataFrame(_rows)
_out = "ACP_avec_reentrainement.xlsx"
_df.to_excel(_out, index=False, sheet_name="PCA_avec_FT")
print(f"[Export] {_out} généré ({len(_rows) - 1} configurations + ligne référence).")


In [ ]:
criterion = nn.CrossEntropyLoss()

# ══════════════════════════════════════════════════════════════════════════════
# BLOC A — InitialNet de référence (2 epochs)
# ══════════════════════════════════════════════════════════════════════════════
net_ref = InitialNet().to(device)
opt_ref = optim.Adam(net_ref.parameters(), lr=1e-3)
for ep in range(2):
    train_one_epoch(net_ref, trainloader, opt_ref, criterion, device)
_, ref_acc = evaluate(net_ref, testloader, criterion, device)
ref_conv_params = (net_ref.conv1.weight.numel() + net_ref.conv1.bias.numel() +
                   net_ref.conv2.weight.numel() + net_ref.conv2.bias.numel())
ref_conv_bytes  = ref_conv_params * 4
print(f"InitialNet (2 epochs) : acc={ref_acc:.2%} | conv params={ref_conv_params:,}\n")

# ══════════════════════════════════════════════════════════════════════════════
# BLOC B — PCA seule (inférence directe) + PCA avec ré-entraînement (2 epochs)
# Pour chaque eTh : même architecture, mêmes conv_params, même gain
# ══════════════════════════════════════════════════════════════════════════════
eTh_list = [0.25, 0.50, 0.75, 0.90]
table = []

for eTh in eTh_list:
    print(f"\n{'─'*60}")
    print(f"  eTh = {eTh:.0%}")
    print(f"{'─'*60}")

    # ── Approche 1 : PCA seule, inférence directe ─────────────────────────────
    base_noFT = deepcopy(net_ref).cpu()
    anlz_noFT = FilterAnalyzer(energy_threshold=eTh)
    anlz_noFT.analyze(base_noFT.conv1, base_noFT.conv2)
    sel_noFT = PseudoRandomFilterSelector(randomization_ratio=0.0, K=0, seed=0)
    sel_noFT.select(anlz_noFT)
    nrc_noFT = NRCNet(initial_net=base_noFT, analyzer=anlz_noFT,
                      selector=sel_noFT, freeze_fc=False).to(device)
    _, acc_noFT = evaluate(nrc_noFT, testloader, criterion, device)
    print(f"  [Sans FT]  acc={acc_noFT:.2%}")

    # ── Approche 2 : PCA + ré-entraînement partiel (2 epochs depuis zéro) ─────
    base_FT = InitialNet().cpu()          # poids vierges — budget total = 2 epochs
    anlz_FT = FilterAnalyzer(energy_threshold=eTh)
    anlz_FT.analyze(base_FT.conv1, base_FT.conv2)
    sel_FT = PseudoRandomFilterSelector(randomization_ratio=0.0, K=0, seed=0)
    sel_FT.select(anlz_FT)
    nrc_FT = NRCNet(initial_net=base_FT, analyzer=anlz_FT,
                    selector=sel_FT, freeze_fc=False).to(device)
    trainable = [p for p in nrc_FT.parameters() if p.requires_grad]
    opt_ft = optim.Adam(trainable, lr=1e-3)
    for ep in range(1, 3):
        tl, ta = train_one_epoch(nrc_FT, trainloader, opt_ft, criterion, device)
        vl, va = evaluate(nrc_FT, testloader, criterion, device)
        print(f"  [Avec FT]  Ép {ep}/2 | train acc={ta:.2%} | val acc={va:.2%}")
    acc_FT = va

    # ── Métriques conv — calculées une seule fois (identiques pour les deux) ──
    # L'architecture NRC est la même quel que soit l'entraînement :
    # conv_params, conv_bytes et conv_gain ne dépendent que de eTh
    conv_params = (sum(p.numel() for p in nrc_noFT.nrc_conv1.parameters()) +
                   sum(p.numel() for p in nrc_noFT.nrc_conv2.parameters()))
    conv_bytes  = conv_params * 4
    conv_gain   = (1 - conv_params / ref_conv_params) * 100

    table.append({
        "eTh"        : eTh,
        "acc_noFT"   : acc_noFT,
        "acc_FT"     : acc_FT,
        "conv_params": conv_params,
        "conv_bytes" : conv_bytes,
        "conv_gain"  : conv_gain,
        "s1"         : anlz_noFT.conv1["s"],
        "s2"         : anlz_noFT.conv2["s"],
    })

# ── Tableau combiné ───────────────────────────────────────────────────────────
print(f"\n{'eTh':>6} | {'Acc (sans FT)':>13} | {'Acc (avec FT)':>13} | "
      f"{'Conv params':>12} | {'Mémoire conv':>13} | {'Gain conv':>10} | {'s₁':>4} | {'s₂':>4}")
print("─" * 100)
for r in table:
    print(f"{r['eTh']:>5.0%} | {r['acc_noFT']:>13.2%} | {r['acc_FT']:>13.2%} | "
          f"{r['conv_params']:>12,} | {r['conv_bytes']/1024:>11.1f} KB | "
          f"{r['conv_gain']:>+9.1f}% | {r['s1']:>4} | {r['s2']:>4}")
print(f"{'Réf':>6} | {ref_acc:>13.2%} | {ref_acc:>13.2%} | "
      f"{ref_conv_params:>12,} | {ref_conv_bytes/1024:>11.1f} KB | "
      f"{'—':>10} | {'—':>4} | {'—':>4}")

# ── Visualisation ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(11, 15))
fig.suptitle("PCA seule : sans vs avec ré-entraînement (2 epochs)",
             fontsize=14, fontweight="bold")

x      = range(len(table))
width  = 0.35
x_labels = [f"{r['eTh']:.0%}" for r in table]
accs_noFT = [r["acc_noFT"] * 100 for r in table]
accs_FT   = [r["acc_FT"]   * 100 for r in table]
conv_mems = [r["conv_bytes"] / 1024 for r in table]
conv_gains= [r["conv_gain"]         for r in table]

# Graphe 1 : Accuracy — barres côte à côte
bars1a = axes[0].bar([i - width/2 for i in x], accs_noFT, width,
                     label="Sans ré-entraînement", color="#4C72B0",
                     edgecolor="black", linewidth=0.5)
bars1b = axes[0].bar([i + width/2 for i in x], accs_FT, width,
                     label="Avec ré-entraînement", color="#64B5CD",
                     edgecolor="black", linewidth=0.5)
axes[0].axhline(ref_acc * 100, color="red", linestyle="--", linewidth=1.5,
                label=f"Initial ({ref_acc:.2%})")
for bar, val in zip(bars1a, accs_noFT):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 f"{val:.1f}%", ha="center", fontsize=9, fontweight="bold")
for bar, val in zip(bars1b, accs_FT):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 f"{val:.1f}%", ha="center", fontsize=9, fontweight="bold")
axes[0].set_xticks(list(x)); axes[0].set_xticklabels(x_labels)
axes[0].set_xlabel("Seuil d'énergie (eTh)")
axes[0].set_ylabel("Test Accuracy (%)")
axes[0].set_title("Précision — sans vs avec ré-entraînement")
axes[0].legend(); axes[0].grid(axis="y", alpha=0.3)

# Graphe 2 : Taille mémoire conv — une seule barre (identique pour les deux)
bars2 = axes[1].bar(x_labels, conv_mems, color="#55A868",
                    edgecolor="black", linewidth=0.5)
axes[1].axhline(ref_conv_bytes / 1024, color="red", linestyle="--", linewidth=1.5,
                label=f"Conv initial ({ref_conv_bytes/1024:.1f} KB)")
for bar, val, r in zip(bars2, conv_mems, table):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                 f"{val:.2f} KB\n({r['conv_params']:,} params)\ns₁={r['s1']}, s₂={r['s2']}",
                 ha="center", fontsize=8, fontweight="bold")
axes[1].set_xlabel("Seuil d'énergie (eTh)")
axes[1].set_ylabel("Mémoire couches conv (KB)")
axes[1].set_title("Taille mémoire conv — identique sans/avec ré-entraînement")
axes[1].legend(); axes[1].grid(axis="y", alpha=0.3)

# Graphe 3 : Gain de compression conv — une seule barre (identique pour les deux)
bars3 = axes[2].bar(x_labels, conv_gains, color="#C44E52",
                    edgecolor="black", linewidth=0.5)
for bar, val in zip(bars3, conv_gains):
    axes[2].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 f"{val:.1f}%", ha="center", fontsize=10, fontweight="bold")
axes[2].set_xlabel("Seuil d'énergie (eTh)")
axes[2].set_ylabel("Gain de compression (%)")
axes[2].set_title("Gain conv — identique sans/avec ré-entraînement")
axes[2].axhline(0, color="gray", linestyle="-", linewidth=0.5)
axes[2].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ─── Impact de p (pseudo-randomisation) — Étapes 1, 2 et 3 seulement (sans étape 4) ─

criterion = nn.CrossEntropyLoss()

# ── Étape 1 : Entraîner un InitialNet frais sur 2 epochs ──────────────────────
net_base = InitialNet().to(device)
opt_base = optim.Adam(net_base.parameters(), lr=1e-3)
for ep in range(2):
    train_one_epoch(net_base, trainloader, opt_base, criterion, device)
base_loss, base_acc = evaluate(net_base, testloader, criterion, device)
base_params = sum(p.numel() for p in net_base.parameters())
base_bytes = base_params * 4

print(f"InitialNet (2 epochs) : acc={base_acc:.2%} | params={base_params:,}\n")

eTh_list_exp = [0.25, 0.50, 0.75, 0.90]
p_list_exp = [0.00, 0.25, 0.50, 0.75, 0.90]

results_exp = []

for eTh_val in eTh_list_exp:
    for p_val in p_list_exp:
        base_copy = deepcopy(net_base).cpu()

        # ── Étape 2 : PCA avec seuil d'énergie eTh ────────────────────────────
        anlz_exp = FilterAnalyzer(energy_threshold=eTh_val)
        anlz_exp.analyze(base_copy.conv1, base_copy.conv2)

        # ── Étape 3 : Pseudo-randomisation LFSR + Grassmann avec ratio p ──────
        sel_exp = PseudoRandomFilterSelector(randomization_ratio=p_val, K=100, seed=42)
        sel_exp.select(anlz_exp)

        # Construction du NRCNet pour l'évaluation uniquement — aucun ré-entraînement (étape 4 absente)
        nrc_exp = NRCNet(initial_net=base_copy, analyzer=anlz_exp, selector=sel_exp, freeze_fc=False).to(device)

        # Évaluation directe sans ré-entraînement
        t_loss, t_acc = evaluate(nrc_exp, testloader, criterion, device)

        # Paramètres réellement stockés (étapes 1+2+3) :
        #   BNRC_keep (vecteurs PCA conservés) + seeds LFSR + w_mean + conv_coord (Y_NRC) + FC
        #   conv_base non compté car régénérable à partir des seeds
        b1_exp = sel_exp.conv1_bnrc
        b2_exp = sel_exp.conv2_bnrc
        n_p_exp = (
            b1_exp["BNRC_keep"].size + len(b1_exp["selected_seeds"]) + anlz_exp.conv1["w_mean"].size +
            b2_exp["BNRC_keep"].size + len(b2_exp["selected_seeds"]) + anlz_exp.conv2["w_mean"].size +
            sum(p.numel() for p in nrc_exp.nrc_conv1.conv_coord.parameters()) +
            sum(p.numel() for p in nrc_exp.nrc_conv2.conv_coord.parameters()) +
            sum(p.numel() for layer in [nrc_exp.fc1, nrc_exp.fc2, nrc_exp.fc3] for p in layer.parameters())
        )
        mem_exp = n_p_exp * 4
        gain_exp = (1 - n_p_exp / base_params) * 100

        results_exp.append({
            "eTh": eTh_val, "p": p_val, "acc": t_acc, "params": n_p_exp,
            "bytes": mem_exp, "gain": gain_exp,
            "s1": anlz_exp.conv1["s"], "s2": anlz_exp.conv2["s"],
        })

# ─── Tableau récapitulatif ────────────────────────────────────────────────────
print(f"{'eTh':>5} {'p':>5} | {'Accuracy':>10} | {'Params':>10} | {'Mémoire':>10} | {'Gain':>8} | {'s1':>4} {'s2':>4}")
print("─" * 75)
for r in results_exp:
    ms = f"{r['bytes']/1024:.1f} KB"
    print(f"{r['eTh']:>4.0%} {r['p']:>5.2f} | {r['acc']:>9.2%} | {r['params']:>10,} | {ms:>10} | {r['gain']:>+7.1f}% | {r['s1']:>4} {r['s2']:>4}")
print(f"{'Ref':>11} | {base_acc:>9.2%} | {base_params:>10,} | {base_bytes/1024:.1f} KB | {'—':>8} | {'—':>4} {'—':>4}")

# ─── Visualisation ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(12, 16))
fig.suptitle("Impact de p — Étapes 1+2+3 seulement (sans ré-entraînement) — 2 epochs", fontsize=14, fontweight="bold")

colors_map = {0.25: "#4C72B0", 0.50: "#55A868", 0.75: "#C44E52", 0.90: "#8172B3"}

# Graphe 1 : Accuracy vs p
for eTh_val in eTh_list_exp:
    subset = [r for r in results_exp if r["eTh"] == eTh_val]
    ps = [r["p"] for r in subset]
    ac = [r["acc"] * 100 for r in subset]
    axes[0].plot(ps, ac, marker="o", label=f"eTh={eTh_val:.0%}", color=colors_map[eTh_val], linewidth=2)
axes[0].axhline(base_acc * 100, color="red", linestyle="--", linewidth=1.5, label=f"Initial ({base_acc:.2%})")
axes[0].set_xlabel("p (randomization ratio)")
axes[0].set_ylabel("Test Accuracy (%)")
axes[0].set_title("Précision vs p (sans ré-entraînement)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Graphe 2 : Taille mémoire vs p
for eTh_val in eTh_list_exp:
    subset = [r for r in results_exp if r["eTh"] == eTh_val]
    ps = [r["p"] for r in subset]
    me = [r["bytes"] / 1024 for r in subset]
    axes[1].plot(ps, me, marker="s", label=f"eTh={eTh_val:.0%}", color=colors_map[eTh_val], linewidth=2)
axes[1].axhline(base_bytes / 1024, color="red", linestyle="--", linewidth=1.5, label=f"Initial ({base_bytes/1024:.1f} KB)")
axes[1].set_xlabel("p (randomization ratio)")
axes[1].set_ylabel("Taille mémoire stockée (KB)")
axes[1].set_title("Taille mémoire stockée vs p")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Graphe 3 : Gain mémoire vs p
for eTh_val in eTh_list_exp:
    subset = [r for r in results_exp if r["eTh"] == eTh_val]
    ps = [r["p"] for r in subset]
    ga = [r["gain"] for r in subset]
    axes[2].plot(ps, ga, marker="^", label=f"eTh={eTh_val:.0%}", color=colors_map[eTh_val], linewidth=2)
axes[2].axhline(0, color="gray", linestyle="-", linewidth=0.5)
axes[2].set_xlabel("p (randomization ratio)")
axes[2].set_ylabel("Gain de compression (%)")
axes[2].set_title("Réduction des paramètres stockés vs p")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# ── Export Excel : Pseudo-randomisation sans ré-entraînement ─────────────────
import pandas as pd

_rows = []
for r in results_exp:
    _rows.append({
        "eTh": f"{r['eTh']:.0%}",
        "p (randomization ratio)": r["p"],
        "Accuracy (%)": round(r["acc"] * 100, 2),
        "Params stockés": r["params"],
        "Mémoire (KB)": round(r["bytes"] / 1024, 2),
        "Gain compression (%)": round(r["gain"], 2),
        "s_conv1": r["s1"],
        "s_conv2": r["s2"],
        "Ré-entraînement": "Non",
    })
_rows.append({
    "eTh": "Référence",
    "p (randomization ratio)": "—",
    "Accuracy (%)": round(base_acc * 100, 2),
    "Params stockés": base_params,
    "Mémoire (KB)": round(base_bytes / 1024, 2),
    "Gain compression (%)": None,
    "s_conv1": "—",
    "s_conv2": "—",
    "Ré-entraînement": "—",
})
_df = pd.DataFrame(_rows)
_out = "Pseudo_sans_reentrainement.xlsx"
_df.to_excel(_out, index=False, sheet_name="Pseudo_sans_FT")
print(f"[Export] {_out} généré ({len(_rows) - 1} configurations + ligne référence).")


In [ ]:
import matplotlib.pyplot as plt

# Pour chaque eTh : (nb_PCA_conservés, distance_Grassmann)
# Les distances sont triées par nb_PCA_conservés décroissant pour voir la décroissance
data_decroissant = {
    25: ([3, 2], [0.000, 1.345]),
    50: ([7, 6, 3, 2], [0.000, 1.357, 2.237, 2.540]),
    75: ([13, 10, 6, 3, 2], [0.000, 1.827, 2.726, 3.199, 3.450]),
    90: ([17, 13, 8, 4, 2], [0.000, 2.248, 3.070, 3.636, 3.933]),
}

plt.figure(figsize=(8,5))
colors = {25: 'blue', 50: 'green', 75: 'orange', 90: 'red'}

for eth, (x, y) in data_decroissant.items():
    plt.plot(x, y, marker='o', linestyle='-', color=colors[eth], label=f'eTh = {eth}%')

# Inverser l'axe des X pour que la lecture aille de "beaucoup de PCA" à "peu de PCA"
plt.gca().invert_xaxis()
plt.xlabel('Nombre de vecteurs PCA conservés (→ décroissant)')
plt.ylabel('Distance de Grassmann')
plt.title('Décroissance de la distance de Grassmann\nquand on conserve plus de vecteurs PCA')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8,5))
for eth, (x, y) in data_decroissant.items():
    plt.plot(x, y, marker='o', linestyle='-', color=colors[eth], label=f'eTh = {eth}%')
plt.xlabel('Nombre de vecteurs PCA conservés')
plt.ylabel('Distance de Grassmann')
plt.title('La distance diminue quand on conserve plus de PCA')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ─── Impact de p — Étapes 1+2+3 sans ré-entraînement ET avec ré-entraînement ─

criterion = nn.CrossEntropyLoss()

eTh_list_exp = [0.25, 0.50, 0.75, 0.90]
p_list_exp   = [0.00, 0.25, 0.50, 0.75, 0.90]

# ══════════════════════════════════════════════════════════════════════════════
# BLOC A — InitialNet de référence (2 epochs)
# ══════════════════════════════════════════════════════════════════════════════
net_base = InitialNet().to(device)
opt_base = optim.Adam(net_base.parameters(), lr=1e-3)
for ep in range(2):
    train_one_epoch(net_base, trainloader, opt_base, criterion, device)
_, base_acc = evaluate(net_base, testloader, criterion, device)
base_params = sum(p.numel() for p in net_base.parameters())
base_bytes  = base_params * 4

# ── Référence conv uniquement ─────────────────────────────────────────────────
ref_conv_params = (net_base.conv1.weight.numel() + net_base.conv1.bias.numel() +
                   net_base.conv2.weight.numel() + net_base.conv2.bias.numel())
ref_conv_bytes  = ref_conv_params * 4
print(f"InitialNet (2 epochs) : acc={base_acc:.2%} | "
      f"conv params={ref_conv_params:,} | conv mém={ref_conv_bytes/1024:.1f} KB\n")

# ══════════════════════════════════════════════════════════════════════════════
# BLOC B — Boucle sur (eTh, p)
# ══════════════════════════════════════════════════════════════════════════════
results_noFT = []
results_FT   = []

for eTh_val in eTh_list_exp:
    for p_val in p_list_exp:
        print(f"\n{'─'*60}")
        print(f"  eTh={eTh_val:.0%} | p={p_val:.2f}")
        print(f"{'─'*60}")

        # ── Métriques architecturales conv — calculées une seule fois ──────────
        base_arch = deepcopy(net_base).cpu()
        anlz_arch = FilterAnalyzer(energy_threshold=eTh_val)
        anlz_arch.analyze(base_arch.conv1, base_arch.conv2)
        sel_arch  = PseudoRandomFilterSelector(randomization_ratio=p_val, K=100, seed=42)
        sel_arch.select(anlz_arch)
        nrc_arch  = NRCNet(initial_net=base_arch, analyzer=anlz_arch,
                           selector=sel_arch, freeze_fc=False)

        # Conv params = nrc_conv1 + nrc_conv2 uniquement
        conv_params = (sum(p.numel() for p in nrc_arch.nrc_conv1.parameters()) +
                       sum(p.numel() for p in nrc_arch.nrc_conv2.parameters()))
        conv_bytes  = conv_params * 4
        conv_gain   = (1 - conv_params / ref_conv_params) * 100
        s1          = anlz_arch.conv1["s"]
        s2          = anlz_arch.conv2["s"]

        # ── Approche 1 : inférence directe (sans ré-entraînement) ─────────────
        base_noFT = deepcopy(net_base).cpu()
        anlz_noFT = FilterAnalyzer(energy_threshold=eTh_val)
        anlz_noFT.analyze(base_noFT.conv1, base_noFT.conv2)
        sel_noFT  = PseudoRandomFilterSelector(randomization_ratio=p_val, K=100, seed=42)
        sel_noFT.select(anlz_noFT)
        nrc_noFT  = NRCNet(initial_net=base_noFT, analyzer=anlz_noFT,
                           selector=sel_noFT, freeze_fc=False).to(device)
        _, acc_noFT = evaluate(nrc_noFT, testloader, criterion, device)
        print(f"  [Sans FT]  acc={acc_noFT:.2%}")

        results_noFT.append({
            "eTh": eTh_val, "p": p_val, "acc": acc_noFT,
            "conv_params": conv_params, "conv_bytes": conv_bytes,
            "conv_gain": conv_gain, "s1": s1, "s2": s2,
        })

        # ── Approche 2 : ré-entraînement partiel (2 epochs depuis zéro) ───────
        base_FT = InitialNet().cpu()
        anlz_FT = FilterAnalyzer(energy_threshold=eTh_val)
        anlz_FT.analyze(base_FT.conv1, base_FT.conv2)
        sel_FT  = PseudoRandomFilterSelector(randomization_ratio=p_val, K=100, seed=42)
        sel_FT.select(anlz_FT)
        nrc_FT  = NRCNet(initial_net=base_FT, analyzer=anlz_FT,
                         selector=sel_FT, freeze_fc=False).to(device)
        trainable = [p for p in nrc_FT.parameters() if p.requires_grad]
        opt_ft    = optim.Adam(trainable, lr=1e-3)
        for ep in range(1, 3):
            tl, ta = train_one_epoch(nrc_FT, trainloader, opt_ft, criterion, device)
            vl, va = evaluate(nrc_FT, testloader, criterion, device)
            print(f"  [Avec FT]  Ép {ep}/2 | train acc={ta:.2%} | val acc={va:.2%}")
        acc_FT = va

        results_FT.append({
            "eTh": eTh_val, "p": p_val, "acc": acc_FT,
            "conv_params": conv_params, "conv_bytes": conv_bytes,  # ← identiques
            "conv_gain": conv_gain, "s1": s1, "s2": s2,           # ← identiques
        })

# ── Tableau combiné ───────────────────────────────────────────────────────────
print(f"\n{'eTh':>5} {'p':>5} | {'Acc (sans FT)':>13} | {'Acc (avec FT)':>13} | "
      f"{'Conv params':>12} | {'Mémoire conv':>12} | {'Gain conv':>9} | {'s1':>4} {'s2':>4}")
print("─" * 100)
for r_no, r_ft in zip(results_noFT, results_FT):
    print(f"{r_no['eTh']:>4.0%} {r_no['p']:>5.2f} | {r_no['acc']:>13.2%} | {r_ft['acc']:>13.2%} | "
          f"{r_no['conv_params']:>12,} | {r_no['conv_bytes']/1024:>10.1f} KB | "
          f"{r_no['conv_gain']:>+8.1f}% | {r_no['s1']:>4} {r_no['s2']:>4}")
print(f"{'Ref':>11} | {base_acc:>13.2%} | {base_acc:>13.2%} | "
      f"{ref_conv_params:>12,} | {ref_conv_bytes/1024:>10.1f} KB | "
      f"{'—':>9} | {'—':>4} {'—':>4}")

# ── Visualisation ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(12, 16))
fig.suptitle("Impact de p — Étapes 1+2+3 : sans vs avec ré-entraînement (2 epochs)",
             fontsize=14, fontweight="bold")

colors_map = {0.25: "#4C72B0", 0.50: "#55A868", 0.75: "#C44E52", 0.90: "#8172B3"}

# Graphe 1 : Accuracy vs p — deux courbes par eTh
for eTh_val in eTh_list_exp:
    col    = colors_map[eTh_val]
    sub_no = [r for r in results_noFT if r["eTh"] == eTh_val]
    sub_ft = [r for r in results_FT   if r["eTh"] == eTh_val]
    ps     = [r["p"] for r in sub_no]
    axes[0].plot(ps, [r["acc"] * 100 for r in sub_no],
                 marker="o", color=col, linewidth=2,
                 label=f"eTh={eTh_val:.0%} sans FT")
    axes[0].plot(ps, [r["acc"] * 100 for r in sub_ft],
                 marker="s", color=col, linewidth=2, linestyle="--",
                 label=f"eTh={eTh_val:.0%} avec FT")
axes[0].axhline(base_acc * 100, color="red", linestyle="--", linewidth=1.5,
                label=f"Initial ({base_acc:.2%})")
axes[0].set_xlabel("p (randomization ratio)")
axes[0].set_ylabel("Test Accuracy (%)")
axes[0].set_title("Précision vs p — sans FT (trait plein) vs avec FT (tirets)")
axes[0].legend(ncol=2, fontsize=8)
axes[0].grid(True, alpha=0.3)

# Graphe 2 : Taille mémoire conv vs p — une seule courbe par eTh
for eTh_val in eTh_list_exp:
    subset = [r for r in results_noFT if r["eTh"] == eTh_val]
    ps     = [r["p"]               for r in subset]
    mems   = [r["conv_bytes"]/1024 for r in subset]
    axes[1].plot(ps, mems, marker="s", label=f"eTh={eTh_val:.0%}",
                 color=colors_map[eTh_val], linewidth=2)
    for px, my, r in zip(ps, mems, subset):
        axes[1].annotate(f"{my:.2f} KB\n({r['conv_params']:,})\ns₁={r['s1']},s₂={r['s2']}",
                         xy=(px, my), xytext=(0, 10),
                         textcoords="offset points", ha="center",
                         fontsize=6.5, fontweight="bold")
axes[1].axhline(ref_conv_bytes / 1024, color="red", linestyle="--", linewidth=1.5,
                label=f"Conv initial ({ref_conv_bytes/1024:.1f} KB)")
axes[1].set_xlabel("p (randomization ratio)")
axes[1].set_ylabel("Mémoire couches conv (KB)")
axes[1].set_title("Taille mémoire conv vs p — identique sans/avec ré-entraînement")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Graphe 3 : Gain conv vs p — une seule courbe par eTh
for eTh_val in eTh_list_exp:
    subset = [r for r in results_noFT if r["eTh"] == eTh_val]
    ps     = [r["p"]          for r in subset]
    gains  = [r["conv_gain"]  for r in subset]
    axes[2].plot(ps, gains, marker="^", label=f"eTh={eTh_val:.0%}",
                 color=colors_map[eTh_val], linewidth=2)
    for px, gy in zip(ps, gains):
        axes[2].annotate(f"{gy:.1f}%", xy=(px, gy), xytext=(0, 8),
                         textcoords="offset points", ha="center",
                         fontsize=7, fontweight="bold")
axes[2].axhline(0, color="gray", linestyle="-", linewidth=0.5)
axes[2].set_xlabel("p (randomization ratio)")
axes[2].set_ylabel("Gain de compression conv (%)")
axes[2].set_title("Gain conv vs p — identique sans/avec ré-entraînement")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Export Excel : Pseudo-randomisation avec ré-entraînement (2 feuilles) ────
import pandas as pd

def _build_pseudo_rows(results, ref_acc_val, ref_conv_params_val, ref_conv_bytes_val, retrain_label):
    rows = []
    for r in results:
        rows.append({
            "eTh": f"{r['eTh']:.0%}",
            "p (randomization ratio)": r["p"],
            "Accuracy (%)": round(r["acc"] * 100, 2),
            "Conv params": r["conv_params"],
            "Mémoire conv (KB)": round(r["conv_bytes"] / 1024, 2),
            "Gain compression (%)": round(r["conv_gain"], 2),
            "s_conv1": r["s1"],
            "s_conv2": r["s2"],
            "Ré-entraînement": retrain_label,
        })
    rows.append({
        "eTh": "Référence",
        "p (randomization ratio)": "—",
        "Accuracy (%)": round(ref_acc_val * 100, 2),
        "Conv params": ref_conv_params_val,
        "Mémoire conv (KB)": round(ref_conv_bytes_val / 1024, 2),
        "Gain compression (%)": None,
        "s_conv1": "—",
        "s_conv2": "—",
        "Ré-entraînement": "—",
    })
    return rows

_rows_noFT = _build_pseudo_rows(results_noFT, base_acc, ref_conv_params, ref_conv_bytes, "Non")
_rows_FT   = _build_pseudo_rows(results_FT,   base_acc, ref_conv_params, ref_conv_bytes, "Oui")

_out = "Pseudo_avec_reentrainement.xlsx"
with pd.ExcelWriter(_out, engine="openpyxl") as writer:
    pd.DataFrame(_rows_noFT).to_excel(writer, index=False, sheet_name="Sans_reentrainement")
    pd.DataFrame(_rows_FT).to_excel(writer,   index=False, sheet_name="Avec_reentrainement")

print(f"[Export] {_out} généré (2 feuilles : Sans_reentrainement + Avec_reentrainement).")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

eTh_list_exp = [0.25, 0.50, 0.75, 0.90]
p_list_exp   = [0.00, 0.25, 0.50, 0.75, 0.90]
eTh_labels   = ['25%', '50%', '75%', '90%']
p_labels     = ['p=0.00', 'p=0.25', 'p=0.50', 'p=0.75', 'p=0.90']

# ── Extraire les matrices depuis results_noFT et results_FT ───────────────────
acc_noFT_mat  = np.array([[r["acc"] * 100
                           for r in results_noFT if r["eTh"] == e]
                           for e in eTh_list_exp])
acc_FT_mat    = np.array([[r["acc"] * 100
                           for r in results_FT   if r["eTh"] == e]
                           for e in eTh_list_exp])
mem_mat       = np.array([[r["conv_bytes"] / 1024
                           for r in results_noFT if r["eTh"] == e]
                           for e in eTh_list_exp])
gain_mat      = np.array([[r["conv_gain"]
                           for r in results_noFT if r["eTh"] == e]
                           for e in eTh_list_exp])

cmap_rg = mcolors.LinearSegmentedColormap.from_list(
    "red_green", ["#E34A4A", "#F5C842", "#55A868"])
cmap_gr = mcolors.LinearSegmentedColormap.from_list(
    "green_red", ["#55A868", "#F5C842", "#E34A4A"])

def annotate_heatmap(ax, mat, fmt, vmin, vmax):
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat[i, j]
            t   = (val - vmin) / (vmax - vmin) if vmax > vmin else 0.5
            col = "white" if t < 0.25 or t > 0.75 else "black"
            ax.text(j, i, fmt(val), ha="center", va="center",
                    fontsize=9, fontweight="bold", color=col)

fig, axes = plt.subplots(3, 1, figsize=(10, 11))
fig.suptitle("Impact de p — NRC conv : précision, mémoire et gain",
             fontsize=13, fontweight="bold")

# ── Tableau 1 : Précision (sans FT en haut, avec FT en bas dans chaque cellule)
ax = axes[0]
acc_vmin = min(acc_noFT_mat.min(), acc_FT_mat.min())
acc_vmax = max(acc_noFT_mat.max(), acc_FT_mat.max())

# Matrice combinée : lignes alternées noFT / FT
n_eth = len(eTh_list_exp)
acc_combined = np.empty((n_eth * 2, len(p_list_exp)))
row_labels   = []
for i, e in enumerate(eTh_list_exp):
    acc_combined[2*i]   = acc_noFT_mat[i]
    acc_combined[2*i+1] = acc_FT_mat[i]
    row_labels += [f"{eTh_labels[i]}\nsans FT", f"{eTh_labels[i]}\navec FT"]

im0 = ax.imshow(acc_combined, cmap=cmap_rg,
                vmin=acc_vmin, vmax=acc_vmax, aspect="auto")
ax.set_xticks(range(len(p_list_exp))); ax.set_xticklabels(p_labels, fontsize=9)
ax.set_yticks(range(len(row_labels))); ax.set_yticklabels(row_labels, fontsize=8)

# Séparateurs entre groupes eTh
for i in range(1, n_eth):
    ax.axhline(2*i - 0.5, color="white", linewidth=2)

annotate_heatmap(ax, acc_combined,
                 lambda v: f"{v:.2f}%", acc_vmin, acc_vmax)
ax.set_title("Précision test (%) — sans FT / avec FT", fontsize=11)
fig.colorbar(im0, ax=ax, fraction=0.03, pad=0.02, label="Accuracy (%)")

# ── Tableau 2 : Mémoire conv ──────────────────────────────────────────────────
ax = axes[1]
mem_vmin, mem_vmax = mem_mat.min(), mem_mat.max()
im1 = ax.imshow(mem_mat, cmap=cmap_gr,
                vmin=mem_vmin, vmax=mem_vmax, aspect="auto")
ax.set_xticks(range(len(p_list_exp))); ax.set_xticklabels(p_labels, fontsize=9)
ax.set_yticks(range(n_eth));           ax.set_yticklabels(eTh_labels, fontsize=9)
annotate_heatmap(ax, mem_mat,
                 lambda v: f"{v:.1f} KB", mem_vmin, mem_vmax)
ax.set_title("Mémoire conv (KB) — identique sans/avec FT", fontsize=11)
fig.colorbar(im1, ax=ax, fraction=0.03, pad=0.02, label="KB")

# ── Tableau 3 : Gain de compression conv ─────────────────────────────────────
ax = axes[2]
gain_vmin, gain_vmax = gain_mat.min(), gain_mat.max()
im2 = ax.imshow(gain_mat, cmap=cmap_rg,
                vmin=gain_vmin, vmax=gain_vmax, aspect="auto")
ax.set_xticks(range(len(p_list_exp))); ax.set_xticklabels(p_labels, fontsize=9)
ax.set_yticks(range(n_eth));           ax.set_yticklabels(eTh_labels, fontsize=9)
annotate_heatmap(ax, gain_mat,
                 lambda v: f"+{v:.1f}%", gain_vmin, gain_vmax)
ax.set_title("Gain de compression conv (%) — identique sans/avec FT", fontsize=11)
fig.colorbar(im2, ax=ax, fraction=0.03, pad=0.02, label="Gain (%)")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# ── Construction des DataFrames depuis results_noFT et results_FT ─────────────
df_noFT = pd.DataFrame(results_noFT)
df_FT   = pd.DataFrame(results_FT)

def make_pivot(df, col, mult=1):
    pv = df.pivot(index="eTh", columns="p", values=col) * mult
    pv.index   = [f"{v:.0%}" for v in pv.index]
    pv.columns = [f"p={v:.2f}" for v in pv.columns]
    pv.index.name = "eTh \\ p"
    return pv

# ── Précision ─────────────────────────────────────────────────────────────────
pivot_acc_noFT = make_pivot(df_noFT, "acc", 100)
pivot_acc_FT   = make_pivot(df_FT,   "acc", 100)

# ── Mémoire conv (KB) — identique sans/avec FT ────────────────────────────────
pivot_mem = make_pivot(df_noFT, "conv_bytes", 1/1024)

# ── Gain conv (%) — identique sans/avec FT ────────────────────────────────────
pivot_gain = make_pivot(df_noFT, "conv_gain")

# ══════════════════════════════════════════════════════════════════════════════
# Affichage
# ══════════════════════════════════════════════════════════════════════════════

# Plage commune pour les deux pivots accuracy (même échelle de couleur)
acc_min = min(pivot_acc_noFT.min().min(), pivot_acc_FT.min().min())
acc_max = max(pivot_acc_noFT.max().max(), pivot_acc_FT.max().max())

print("━" * 70)
print("  Précision test (%) — sans ré-entraînement (inférence directe)")
print(f"  Référence InitialNet : {base_acc:.2%}")
print("━" * 70)
display(pivot_acc_noFT.style
    .format("{:.2f}%")
    .background_gradient(cmap="RdYlGn", axis=None,
                         vmin=acc_min, vmax=acc_max)
    .set_caption("Précision Test (%) — sans FT"))

print()
print("━" * 70)
print("  Précision test (%) — avec ré-entraînement partiel (2 epochs)")
print(f"  Référence InitialNet : {base_acc:.2%}")
print("━" * 70)
display(pivot_acc_FT.style
    .format("{:.2f}%")
    .background_gradient(cmap="RdYlGn", axis=None,
                         vmin=acc_min, vmax=acc_max)
    .set_caption("Précision Test (%) — avec FT"))

print()
print("━" * 70)
print("  Mémoire conv (KB) — identique sans/avec FT")
print(f"  Référence conv initiales : {base_acc:.2%} | {ref_conv_bytes/1024:.1f} KB")
print("━" * 70)
display(pivot_mem.style
    .format("{:.1f} KB")
    .background_gradient(cmap="RdYlGn_r", axis=None)
    .set_caption("Mémoire conv (KB) — identique sans/avec FT"))

print()
print("━" * 70)
print("  Gain de compression conv (%) — identique sans/avec FT")
print("━" * 70)
display(pivot_gain.style
    .format("{:+.1f}%")
    .background_gradient(cmap="RdYlGn", axis=None)
    .set_caption("Gain de compression conv (%) — identique sans/avec FT"))

In [ ]:
# ─── Impact de p avec ré-entraînement partiel (étapes 1+2+3+4) ───────────────

criterion_full = nn.CrossEntropyLoss()

eTh_values_full = [0.25, 0.50, 0.75, 0.90]
p_values_full = [0.00, 0.25, 0.50, 0.75, 0.90]

results_full = []

for eTh_val in eTh_values_full:
    # Étape 1 : entraîner InitialNet frais (2 epochs)
    net_init_full = InitialNet().to(device)
    opt_init_full = optim.Adam(net_init_full.parameters(), lr=1e-3)
    for ep in range(2):
        train_one_epoch(net_init_full, trainloader, opt_init_full, criterion_full, device)
    init_loss_full, init_acc_full = evaluate(net_init_full, testloader, criterion_full, device)
    init_params_full = sum(p.numel() for p in net_init_full.parameters())
    print(f"\n[eTh={eTh_val:.0%}] InitialNet : acc={init_acc_full:.2%} | params={init_params_full:,}")

    for p_val in p_values_full:
        base_cp = deepcopy(net_init_full).cpu()

        # Étape 2 : PCA
        anlz_f = FilterAnalyzer(energy_threshold=eTh_val)
        anlz_f.analyze(base_cp.conv1, base_cp.conv2)

        # Étape 3 : Pseudo-randomisation
        sel_f = PseudoRandomFilterSelector(randomization_ratio=p_val, K=100, seed=42)
        sel_f.select(anlz_f)

        # Étape 4 : Construction NRCNet + ré-entraînement partiel (2 epochs)
        nrc_f = NRCNet(initial_net=base_cp, analyzer=anlz_f, selector=sel_f, freeze_fc=False).to(device)
        opt_nrc_f = optim.Adam(filter(lambda param: param.requires_grad, nrc_f.parameters()), lr=1e-3)
        for ep in range(2):
            train_one_epoch(nrc_f, trainloader, opt_nrc_f, criterion_full, device)
        nrc_loss_f, nrc_acc_f = evaluate(nrc_f, testloader, criterion_full, device)

        # Paramètres stockés
        b1_f = sel_f.conv1_bnrc
        b2_f = sel_f.conv2_bnrc
        n_stored = (
            b1_f["BNRC_keep"].size + len(b1_f["selected_seeds"]) + anlz_f.conv1["w_mean"].size +
            b2_f["BNRC_keep"].size + len(b2_f["selected_seeds"]) + anlz_f.conv2["w_mean"].size +
            sum(p.numel() for p in nrc_f.nrc_conv1.conv_coord.parameters()) +
            sum(p.numel() for p in nrc_f.nrc_conv2.conv_coord.parameters()) +
            sum(p.numel() for layer in [nrc_f.fc1, nrc_f.fc2, nrc_f.fc3] for p in layer.parameters())
        )
        mem_f = n_stored * 4
        gain_f = (1 - n_stored / init_params_full) * 100

        results_full.append({
            "eTh": eTh_val, "p": p_val,
            "init_acc": init_acc_full, "nrc_acc": nrc_acc_f,
            "delta_acc": nrc_acc_f - init_acc_full,
            "params": n_stored, "bytes": mem_f, "gain": gain_f,
            "s1": anlz_f.conv1["s"], "s2": anlz_f.conv2["s"],
        })
        print(f"  p={p_val:.2f} | nrc_acc={nrc_acc_f:.2%} | Δ={nrc_acc_f - init_acc_full:+.2%} | gain={gain_f:+.1f}%")

# ─── Tableau récapitulatif ────────────────────────────────────────────────────
print(f"\n{'eTh':>5} {'p':>5} | {'Init%':>7} {'NRC%':>7} {'Δ%':>7} | {'Params':>10} | {'Mémoire':>10} | {'Gain':>8}")
print("─" * 80)
for r in results_full:
    ms = f"{r['bytes']/1024:.1f} KB"
    print(f"{r['eTh']:>4.0%} {r['p']:>5.2f} | {r['init_acc']:>6.2%} {r['nrc_acc']:>6.2%} {r['delta_acc']:>+6.2%} | {r['params']:>10,} | {ms:>10} | {r['gain']:>+7.1f}%")

# ─── Tableaux pivots ─────────────────────────────────────────────────────────
df_full = pd.DataFrame(results_full)

pivot_acc_full = df_full.pivot(index="eTh", columns="p", values="nrc_acc") * 100
pivot_acc_full.index = [f"{v:.0%}" for v in pivot_acc_full.index]
pivot_acc_full.columns = [f"p={v:.2f}" for v in pivot_acc_full.columns]

pivot_mem_full = df_full.pivot(index="eTh", columns="p", values="bytes") / 1024
pivot_mem_full.index = [f"{v:.0%}" for v in pivot_mem_full.index]
pivot_mem_full.columns = [f"p={v:.2f}" for v in pivot_mem_full.columns]

pivot_gain_full = df_full.pivot(index="eTh", columns="p", values="gain")
pivot_gain_full.index = [f"{v:.0%}" for v in pivot_gain_full.index]
pivot_gain_full.columns = [f"p={v:.2f}" for v in pivot_gain_full.columns]

print("\n" + "━" * 70)
print("  Précision NRC (%) — étapes 1+2+3+4 (avec ré-entraînement)")
print("━" * 70)
display(pivot_acc_full.style.format("{:.2f}%").background_gradient(cmap="RdYlGn", axis=None).set_caption("Précision Test (%)"))

print("\n" + "━" * 70)
print("  Taille mémoire stockée (KB)")
print("━" * 70)
display(pivot_mem_full.style.format("{:.1f} KB").background_gradient(cmap="RdYlGn_r", axis=None).set_caption("Mémoire stockée (KB)"))

print("\n" + "━" * 70)
print("  Gain de compression (%)")
print("━" * 70)
display(pivot_gain_full.style.format("{:+.1f}%").background_gradient(cmap="RdYlGn", axis=None).set_caption("Gain de compression (%)"))

# ─── Visualisation ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(12, 16))
fig.suptitle("Impact de p — Étapes 1+2+3+4 (avec ré-entraînement partiel, 2 epochs)", fontsize=14, fontweight="bold")

colors_full = {0.25: "#4C72B0", 0.50: "#55A868", 0.75: "#C44E52", 0.90: "#8172B3"}

for eTh_val in eTh_values_full:
    sub = [r for r in results_full if r["eTh"] == eTh_val]
    ps = [r["p"] for r in sub]
    axes[0].plot(ps, [r["nrc_acc"] * 100 for r in sub], marker="o", label=f"eTh={eTh_val:.0%}", color=colors_full[eTh_val], linewidth=2)
    axes[1].plot(ps, [r["bytes"] / 1024 for r in sub], marker="s", label=f"eTh={eTh_val:.0%}", color=colors_full[eTh_val], linewidth=2)
    axes[2].plot(ps, [r["gain"] for r in sub], marker="^", label=f"eTh={eTh_val:.0%}", color=colors_full[eTh_val], linewidth=2)

# Lignes de référence (moyenne des init_acc)
mean_init = df_full.groupby("eTh")["init_acc"].first().mean() * 100
axes[0].axhline(mean_init, color="red", linestyle="--", linewidth=1.5, label=f"Init moy. ({mean_init:.1f}%)")

for ax, ylabel, title in zip(axes,
    ["Test Accuracy (%)", "Mémoire stockée (KB)", "Gain compression (%)"],
    ["Précision NRC vs p", "Taille mémoire vs p", "Gain compression vs p"]):
    ax.set_xlabel("p (randomization ratio)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

axes[2].axhline(0, color="gray", linestyle="-", linewidth=0.5)

plt.tight_layout()
plt.savefig("nrc_full_pipeline_sweep.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figure sauvegardée → nrc_full_pipeline_sweep.png")

In [ ]:
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    criterion = nn.CrossEntropyLoss()
    initial_net = InitialNet().to(device)
    optimizer = optim.Adam(initial_net.parameters(), lr=1e-3)
    print("[Main] Modèle initial créé.")

    energy_threshold = 0.5  # conserver tous les vecteurs PCA (pas de réduction) pour l'analyse comparative

    for epoch in range(1, 3):
        train_loss, train_acc = train_one_epoch(
            initial_net, trainloader, optimizer, criterion, device
        )
        val_loss, val_acc = evaluate(initial_net, testloader, criterion, device) 

        print(f"\nEpoch {epoch}/2")
        print(f"  InitialNet train | loss={train_loss:.4f} | acc={train_acc:.2%}")
        print(f"  InitialNet test  | loss={val_loss:.4f} | acc={val_acc:.2%}")

        base_for_pca = deepcopy(initial_net).cpu()

        # Étape 2 : PCA seule (eTh)
        analyzer_pca = FilterAnalyzer(energy_threshold=energy_threshold)
        analyzer_pca.analyze(base_for_pca.conv1, base_for_pca.conv2)

        print("\n  PCA seule (sans entraînement) évaluée. Passons à l'entraînement complet du NRCNet avec PCA...")
    
        # Étape 4 : PCA + entraînement complet du NRCNet (gelant les conv de base, entraînant les conv coord)
        # PCA avec entrainement complet du NRCNet (gelant les conv de base, entraînant les conv coord)
        pca_net1 = NRCNet(
            initial_net=base_for_pca,
            analyzer=analyzer_pca,
            selector=selector_pca,
            freeze_fc=False,
        ).to(device)
        
        optimizer_pca = optim.Adam(pca_net1.parameters(), lr=1e-3)
        for epoch_pca in range(1, 3):
            train_loss_pca, train_acc_pca = train_one_epoch(
                pca_net1, trainloader, optimizer_pca, criterion, device
            )
            val_loss_pca, val_acc_pca = evaluate(pca_net1, testloader, criterion, device)

            print(f"  PCA Net Epoch {epoch_pca}/2")
            print(f"    train | loss={train_loss_pca:.4f} | acc={train_acc_pca:.2%}")
            print(f"    test  | loss={val_loss_pca:.4f} | acc={val_acc_pca:.2%}")
            print(f"    Delta test acc   | {val_acc_pca - val_acc:.2%}")

In [ ]:
# print model's state_dict
print("Model's state_dict:")
for param_tensor in initial_net.state_dict():
    print(param_tensor, "\t", initial_net.state_dict()[param_tensor].size())

### Pipeline complet — méthode reformulée en 4 étapes

Cette partie exécute la méthode complète selon la lecture suivante :

1. entraîner `InitialNet`,
2. appliquer la PCA contrôlée par `eTh`,
3. pseudo-randomiser la base via LFSR + Grassmann,
4. ré-entraîner partiellement le modèle NRC obtenu.

In [ ]:
# Pipeline NRC complet importé depuis pipeline.py :
#   run_nrc_pipeline(train_loader, test_loader, device, ...) → tuple
#
# Enchaîne les 4 étapes + annexe de sauvegarde compressée.
# Voir pipeline.py pour le détail.
print("✓ pipeline.py chargé : run_nrc_pipeline")


In [ ]:

# ─── Lancement du pipeline NRC ────────────────────────────────────────────────

initial_net, analyzer, selector, nrc_net, net_restored = run_nrc_pipeline(
    train_loader  = trainloader,
    test_loader   = testloader,
    device        = str(device),
    # Étape 1 — baseline
    n_filters        = 6,
    epochs_initial   = 2,
    lr_initial       = 1e-3,
    weights_path     = "initial_weights.pth",
    # Étape 2 — PCA (seul paramètre = eTh)
    energy_threshold    = 0.5,   # eTh : conserver 75% de la variance
    # Étape 3 — LFSR/Grassmann (paramètre = p)
    randomization_ratio = 0.75,   # p   : remplacer 75% des vecteurs B_T par LFSR
    K           = 100,            # candidates par vecteur
    pseudo_seed = 42,
    # Étape 4 — fine-tuning
    epochs_finetune = 2,
    lr_finetune     = 1e-3,
    freeze_fc       = False,      # False = entraîner Y_NRC + FC (recommandé)
    # Annexe — compression
    compressed_path = "model_nrc.pth",
)


### Illustration de nombre de poids stockés par classe dans mon architecture neuronale

In [ ]:
import torch
import matplotlib.pyplot as plt

# ─── Fonctions utilitaires ────────────────────────────────────────────────────

def size_in_bytes(num_params, dtype=torch.float32):
    return num_params * torch.tensor([], dtype=dtype).element_size()

def human_readable(size):
    for unit in ["B", "KB", "MB", "GB"]:
        if size < 1024:
            return f"{size:.2f} {unit}"
        size /= 1024


# ─── Récupération des blocs ───────────────────────────────────────────────────

b1 = selector.conv1_bnrc
b2 = selector.conv2_bnrc

# 1. Base Bs stockée
stored_Bs = b1["BNRC_keep"].size + b2["BNRC_keep"].size

# 2. Coordonnées Y_NRC
stored_Ynrc = (
    sum(p.numel() for p in nrc_net.nrc_conv1.conv_coord.parameters()) +
    sum(p.numel() for p in nrc_net.nrc_conv2.conv_coord.parameters())
)

# 3. FC layers
stored_fc = sum(
    p.numel() for layer in [nrc_net.fc1, nrc_net.fc2, nrc_net.fc3]
    for p in layer.parameters()
)

# 4. w_mean
stored_wmean = analyzer.conv1["w_mean"].size + analyzer.conv2["w_mean"].size

# 5. seeds (supposées int32)
stored_seeds = len(b1["selected_seeds"]) + len(b2["selected_seeds"])

stored_autres = stored_fc + stored_wmean + stored_seeds

# ─── Calcul mémoire (en bytes) ────────────────────────────────────────────────

bytes_Bs = size_in_bytes(stored_Bs, torch.float32)
bytes_Ynrc = size_in_bytes(stored_Ynrc, torch.float32)
bytes_fc = size_in_bytes(stored_fc, torch.float32)
bytes_wmean = size_in_bytes(stored_wmean, torch.float32)
bytes_seeds = stored_seeds * 4  # int32

bytes_autres = bytes_fc + bytes_wmean + bytes_seeds
total_bytes = bytes_Bs + bytes_Ynrc + bytes_autres

# ─── Affichage détaillé ───────────────────────────────────────────────────────

print("\n📊 Nombre de paramètres")
print(f"Base Bs      : {stored_Bs:>10,}")
print(f"Y_NRC        : {stored_Ynrc:>10,}")
print(f"Autres       : {stored_autres:>10,}")
print(f"  dont FC    : {stored_fc:>10,}")
print(f"  dont w_mean: {stored_wmean:>10,}")
print(f"  dont seeds : {stored_seeds:>10,}")
print(f"TOTAL        : {stored_Bs + stored_Ynrc + stored_autres:>10,}")

print("\n💾 Taille mémoire réelle")
print(f"Base Bs      : {human_readable(bytes_Bs)}")
print(f"Y_NRC        : {human_readable(bytes_Ynrc)}")
print(f"Autres       : {human_readable(bytes_autres)}")
print(f"  dont FC    : {human_readable(bytes_fc)}")
print(f"  dont w_mean: {human_readable(bytes_wmean)}")
print(f"  dont seeds : {human_readable(bytes_seeds)}")
print(f"TOTAL        : {human_readable(total_bytes)}")

# ─── Diagramme basé sur la mémoire ────────────────────────────────────────────

categories = ["Base Bs\n(BNRC_keep)", "Coordonnées\nY_NRC", "Autres\n(FC + w_mean + seeds)"]
values = [bytes_Bs, bytes_Ynrc, bytes_autres]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(categories, values)

for bar, val in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() * 1.02,
        human_readable(val),
        ha="center", va="bottom", fontsize=11, fontweight="bold",
    )

ax.set_ylabel("Taille mémoire (bytes)", fontsize=12)
ax.set_title("Répartition mémoire — Modèle NRC compressé", fontsize=13, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# ─── Décomposition détaillée : poids du modèle initial vs modèle NRC compressé ──

b1 = selector.conv1_bnrc
b2 = selector.conv2_bnrc

# ═══ MODÈLE INITIAL — décomposition par couche ═══════════════════════════════

init_conv1_w = initial_net.conv1.weight.numel()
init_conv1_b = initial_net.conv1.bias.numel()
init_conv2_w = initial_net.conv2.weight.numel()
init_conv2_b = initial_net.conv2.bias.numel()
init_fc1_w   = initial_net.fc1.weight.numel()
init_fc1_b   = initial_net.fc1.bias.numel()
init_fc2_w   = initial_net.fc2.weight.numel()
init_fc2_b   = initial_net.fc2.bias.numel()
init_fc3_w   = initial_net.fc3.weight.numel()
init_fc3_b   = initial_net.fc3.bias.numel()

init_conv_total = init_conv1_w + init_conv1_b + init_conv2_w + init_conv2_b
init_fc_total   = init_fc1_w + init_fc1_b + init_fc2_w + init_fc2_b + init_fc3_w + init_fc3_b
init_total      = init_conv_total + init_fc_total

print("=" * 70)
print("  MODÈLE INITIAL — Décomposition des poids")
print("=" * 70)
print(f"  conv1 weights : {init_conv1_w:>8,}  (shape {tuple(initial_net.conv1.weight.shape)})")
print(f"  conv1 bias    : {init_conv1_b:>8,}")
print(f"  conv2 weights : {init_conv2_w:>8,}  (shape {tuple(initial_net.conv2.weight.shape)})")
print(f"  conv2 bias    : {init_conv2_b:>8,}")
print(f"  ── sous-total conv : {init_conv_total:>8,}")
print(f"  fc1 weights   : {init_fc1_w:>8,}")
print(f"  fc1 bias      : {init_fc1_b:>8,}")
print(f"  fc2 weights   : {init_fc2_w:>8,}")
print(f"  fc2 bias      : {init_fc2_b:>8,}")
print(f"  fc3 weights   : {init_fc3_w:>8,}")
print(f"  fc3 bias      : {init_fc3_b:>8,}")
print(f"  ── sous-total FC   : {init_fc_total:>8,}")
print(f"  TOTAL INITIAL      : {init_total:>8,}")

# ═══ MODÈLE NRC — décomposition des poids stockés ════════════════════════════

# Base Bs (BNRC_keep) — vecteurs PCA conservés explicitement
nrc_Bs_conv1 = b1["BNRC_keep"].size
nrc_Bs_conv2 = b2["BNRC_keep"].size
nrc_Bs_total = nrc_Bs_conv1 + nrc_Bs_conv2

# Seeds LFSR (int32, régénérables)
nrc_seeds_conv1 = len(b1["selected_seeds"])
nrc_seeds_conv2 = len(b2["selected_seeds"])
nrc_seeds_total = nrc_seeds_conv1 + nrc_seeds_conv2

# w_mean (moyenne des filtres, nécessaire à la reconstruction)
nrc_wmean_conv1 = analyzer.conv1["w_mean"].size
nrc_wmean_conv2 = analyzer.conv2["w_mean"].size
nrc_wmean_total = nrc_wmean_conv1 + nrc_wmean_conv2

# Coordonnées Y_NRC (conv_coord)
nrc_coord_conv1 = sum(p.numel() for p in nrc_net.nrc_conv1.conv_coord.parameters())
nrc_coord_conv2 = sum(p.numel() for p in nrc_net.nrc_conv2.conv_coord.parameters())
nrc_coord_total = nrc_coord_conv1 + nrc_coord_conv2

# FC layers (identiques à l'initial, mais stockées)
nrc_fc1 = sum(p.numel() for p in nrc_net.fc1.parameters())
nrc_fc2 = sum(p.numel() for p in nrc_net.fc2.parameters())
nrc_fc3 = sum(p.numel() for p in nrc_net.fc3.parameters())
nrc_fc_total = nrc_fc1 + nrc_fc2 + nrc_fc3

nrc_conv_stored = nrc_Bs_total + nrc_seeds_total + nrc_wmean_total + nrc_coord_total
nrc_total = nrc_conv_stored + nrc_fc_total

print(f"\n{'=' * 70}")
print("  MODÈLE NRC — Décomposition des poids stockés")
print("=" * 70)
print(f"  [conv1] BNRC_keep (Vs)   : {nrc_Bs_conv1:>8,}  ({b1['n_keep']} vecteurs)")
print(f"  [conv1] Seeds LFSR (Vr)  : {nrc_seeds_conv1:>8,}  ({b1['n_rand']} vecteurs régénérés)")
print(f"  [conv1] w_mean           : {nrc_wmean_conv1:>8,}")
print(f"  [conv1] conv_coord (Y)   : {nrc_coord_conv1:>8,}")
print(f"  [conv2] BNRC_keep (Vs)   : {nrc_Bs_conv2:>8,}  ({b2['n_keep']} vecteurs)")
print(f"  [conv2] Seeds LFSR (Vr)  : {nrc_seeds_conv2:>8,}  ({b2['n_rand']} vecteurs régénérés)")
print(f"  [conv2] w_mean           : {nrc_wmean_conv2:>8,}")
print(f"  [conv2] conv_coord (Y)   : {nrc_coord_conv2:>8,}")
print(f"  ── sous-total conv stocké : {nrc_conv_stored:>8,}")
print(f"  fc1                      : {nrc_fc1:>8,}")
print(f"  fc2                      : {nrc_fc2:>8,}")
print(f"  fc3                      : {nrc_fc3:>8,}")
print(f"  ── sous-total FC          : {nrc_fc_total:>8,}")
print(f"  TOTAL NRC STOCKÉ          : {nrc_total:>8,}")

# ═══ COMPARAISON INITIALE vs NRC ═════════════════════════════════════════════

compression_ratio = 1 - nrc_total / init_total
print(f"\n{'=' * 70}")
print("  COMPARAISON")
print("=" * 70)
print(f"  Paramètres Initial       : {init_total:>8,}")
print(f"  Paramètres NRC stockés   : {nrc_total:>8,}")
print(f"  Réduction                : {compression_ratio:>7.1%}")

# ═══ TAILLE MÉMOIRE (bytes) ══════════════════════════════════════════════════

bytes_per_float = 4  # float32
bytes_per_seed  = 4  # int32

init_bytes = init_total * bytes_per_float
nrc_bytes  = (nrc_total - nrc_seeds_total) * bytes_per_float + nrc_seeds_total * bytes_per_seed

print(f"\n  Mémoire Initial          : {human_readable(init_bytes)}")
print(f"  Mémoire NRC              : {human_readable(nrc_bytes)}")
print(f"  Gain mémoire             : {1 - nrc_bytes / init_bytes:>7.1%}")

# ═══ VISUALISATION ════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Décomposition des poids — Initial vs NRC compressé", fontsize=14, fontweight="bold")

# --- Graphe 1 : Modèle Initial par couche ---
init_labels = ["conv1\n(w+b)", "conv2\n(w+b)", "fc1", "fc2", "fc3"]
init_vals   = [init_conv1_w + init_conv1_b, init_conv2_w + init_conv2_b,
               init_fc1_w + init_fc1_b, init_fc2_w + init_fc2_b, init_fc3_w + init_fc3_b]
colors_init = ["#4C72B0", "#4C72B0", "#8172B3", "#8172B3", "#8172B3"]
bars1 = axes[0].bar(init_labels, init_vals, color=colors_init, edgecolor="black", linewidth=0.5)
for bar, val in zip(bars1, init_vals):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(init_vals) * 0.02,
                 f"{val:,}", ha="center", fontsize=9, fontweight="bold")
axes[0].set_ylabel("Nombre de paramètres")
axes[0].set_title("Modèle Initial\n(tous stockés)")
axes[0].grid(axis="y", alpha=0.3)
axes[0].set_axisbelow(True)

# --- Graphe 2 : Modèle NRC par catégorie ---
nrc_labels = ["BNRC_keep\n(Vs)", "Seeds\nLFSR", "w_mean", "conv_coord\n(Y_NRC)", "FC\n(fc1+fc2+fc3)"]
nrc_vals   = [nrc_Bs_total, nrc_seeds_total, nrc_wmean_total, nrc_coord_total, nrc_fc_total]
colors_nrc = ["#4C72B0", "#C44E52", "#CCB974", "#55A868", "#8172B3"]
bars2 = axes[1].bar(nrc_labels, nrc_vals, color=colors_nrc, edgecolor="black", linewidth=0.5)
for bar, val in zip(bars2, nrc_vals):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(nrc_vals) * 0.02,
                 f"{val:,}", ha="center", fontsize=9, fontweight="bold")
axes[1].set_ylabel("Nombre de paramètres")
axes[1].set_title("Modèle NRC\n(paramètres stockés)")
axes[1].grid(axis="y", alpha=0.3)
axes[1].set_axisbelow(True)

# --- Graphe 3 : Comparaison totale Initial vs NRC (empilé conv / FC) ---
x = np.arange(2)
width = 0.45
conv_vals = [init_conv_total, nrc_conv_stored]
fc_vals   = [init_fc_total, nrc_fc_total]

bars_conv = axes[2].bar(x, conv_vals, width, label="Conv (base + coord + meta)", color="#4C72B0")
bars_fc   = axes[2].bar(x, fc_vals, width, bottom=conv_vals, label="FC (fc1 + fc2 + fc3)", color="#8172B3")

axes[2].set_xticks(x)
axes[2].set_xticklabels(["Initial", "NRC"])
axes[2].set_ylabel("Nombre de paramètres")
axes[2].set_title(f"Total stocké\n(réduction : {compression_ratio:.1%})")
axes[2].legend(loc="upper right")
axes[2].grid(axis="y", alpha=0.3)
axes[2].set_axisbelow(True)

for xi, (cv, fv) in enumerate(zip(conv_vals, fc_vals)):
    axes[2].text(xi, cv + fv + max(cv + fv for cv, fv in zip(conv_vals, fc_vals)) * 0.02,
                 f"{cv + fv:,}", ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
from itertools import product

# ─── Test systématique du pipeline NRC (sans CompressedNet) ───────────────────
# On balaye plusieurs combinaisons (eTh, p) et on sauvegarde chaque NRCNet
# avec torch.save classique.


eTh_values = [0.50, 0.75, 0.90, 0.99]
p_values   = [0.00, 0.25, 0.50, 0.75, 1.00]

results = []

criterion_test = nn.CrossEntropyLoss()

for eTh, p in product(eTh_values, p_values):
    tag = f"eTh={eTh:.2f}_p={p:.2f}"
    print(f"\n{'='*60}")
    print(f"  Pipeline : {tag}")
    print(f"{'='*60}")

    # ── Étape 1 : entraîner InitialNet ────────────────────────────────────
    net_init = InitialNet().to(device)
    opt_init = optim.Adam(net_init.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()

    for ep in range(1, 3):
        tr_loss, tr_acc = train_one_epoch(net_init, trainloader, opt_init, crit, device)
    val_loss_init, val_acc_init = evaluate(net_init, testloader, crit, device)
    print(f"  InitialNet test | loss={val_loss_init:.4f} | acc={val_acc_init:.2%}")

    # ── Étape 2 : PCA (eTh) ──────────────────────────────────────────────
    base_model = deepcopy(net_init).cpu()
    anlz = FilterAnalyzer(energy_threshold=eTh)
    anlz.analyze(base_model.conv1, base_model.conv2)

    # ── Étape 3 : Pseudo-randomisation (p) ───────────────────────────────
    sel = PseudoRandomFilterSelector(randomization_ratio=p, K=100, seed=42)
    sel.select(anlz)

    # ── Étape 4 : Construction NRCNet + ré-entraînement partiel ──────────
    nrc = NRCNet(
        initial_net=base_model,
        analyzer=anlz,
        selector=sel,
        freeze_fc=False,
    ).to(device)

    opt_nrc = optim.Adam(filter(lambda param: param.requires_grad, nrc.parameters()), lr=1e-3)

    for ep in range(1, 3):
        ft_loss, ft_acc = train_one_epoch(nrc, trainloader, opt_nrc, crit, device)
    nrc_val_loss, nrc_val_acc = evaluate(nrc, testloader, crit, device)
    print(f"  NRCNet   test   | loss={nrc_val_loss:.4f} | acc={nrc_val_acc:.2%}")
    print(f"  Delta acc       | {nrc_val_acc - val_acc_init:+.2%}")

    # ── Sauvegarde classique (state_dict) ─────────────────────────────────
    save_path = f"nrc_net_{tag}.pth"
    torch.save(nrc.state_dict(), save_path)
    file_size_kb = os.path.getsize(save_path) / 1024
    print(f"  Sauvegardé → {save_path} ({file_size_kb:.1f} Ko)")

    # ── Vérification du rechargement ──────────────────────────────────────
    nrc_check = NRCNet(
        initial_net=base_model,
        analyzer=anlz,
        selector=sel,
        freeze_fc=False,
    ).to(device)
    nrc_check.load_state_dict(torch.load(save_path, map_location=device))
    check_loss, check_acc = evaluate(nrc_check, testloader, crit, device)
    assert abs(check_acc - nrc_val_acc) < 1e-6, "Mismatch après rechargement !"
    print(f"  ✓ Rechargement vérifié (acc={check_acc:.2%})")

    # ── Infos couches ─────────────────────────────────────────────────────
    b1 = sel.conv1_bnrc
    b2 = sel.conv2_bnrc
    results.append({
        "eTh": eTh, "p": p,
        "init_acc": val_acc_init, "nrc_acc": nrc_val_acc,
        "delta_acc": nrc_val_acc - val_acc_init,
        "conv1_s": anlz.conv1["s"], "conv1_keep": b1["n_keep"], "conv1_rand": b1["n_rand"],
        "conv2_s": anlz.conv2["s"], "conv2_keep": b2["n_keep"], "conv2_rand": b2["n_rand"],
        "file_kb": file_size_kb,
        "path": save_path,
    })

# ─── Tableau récapitulatif ────────────────────────────────────────────────────
print("\n\n" + "=" * 100)
print("  TABLEAU RÉCAPITULATIF")
print("=" * 100)
print(f"{'eTh':>5} {'p':>5} | {'Init%':>7} {'NRC%':>7} {'Δ%':>7} | "
      f"{'c1_s':>4} {'c1_k':>4} {'c1_r':>4} | "
      f"{'c2_s':>4} {'c2_k':>4} {'c2_r':>4} | {'Ko':>8}")
print("─" * 100)
for r in results:
    print(f"{r['eTh']:5.2f} {r['p']:5.2f} | "
          f"{r['init_acc']:6.2%} {r['nrc_acc']:6.2%} {r['delta_acc']:+6.2%} | "
          f"{r['conv1_s']:4d} {r['conv1_keep']:4d} {r['conv1_rand']:4d} | "
          f"{r['conv2_s']:4d} {r['conv2_keep']:4d} {r['conv2_rand']:4d} | "
          f"{r['file_kb']:7.1f}")

# ─── Visualisation ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Pipeline NRC — Balayage (eTh, p)", fontsize=14, fontweight="bold")

for eTh_val in eTh_values:
    subset = [r for r in results if r["eTh"] == eTh_val]
    ps   = [r["p"] for r in subset]
    accs = [r["nrc_acc"] * 100 for r in subset]
    axes[0].plot(ps, accs, marker="o", label=f"eTh={eTh_val:.2f}")

axes[0].set_xlabel("p (randomization ratio)")
axes[0].set_ylabel("NRC test accuracy (%)")
axes[0].set_title("Précision NRC vs p")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for eTh_val in eTh_values:
    subset = [r for r in results if r["eTh"] == eTh_val]
    ps   = [r["p"] for r in subset]
    sizes = [r["file_kb"] for r in subset]
    axes[1].plot(ps, sizes, marker="s", label=f"eTh={eTh_val:.2f}")

axes[1].set_xlabel("p (randomization ratio)")
axes[1].set_ylabel("Taille fichier (Ko)")
axes[1].set_title("Taille sauvegardée vs p")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("nrc_sweep_results.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figure sauvegardée → nrc_sweep_results.png")

### Résultats, lecture des paramètres et interprétation

Les sorties ci-dessous servent à relire la méthode sous trois angles :

- performance du modèle initial et du modèle NRC,
- paramètres réellement entraînés ou stockés,
- effet concret des étapes PCA et pseudo-randomization sur la représentation finale.

In [ ]:
# ─── Résumé des résultats NRC ─────────────────────────────────────────────────
import io

print("=" * 70)
print("  RÉSUMÉ NRC")
print("=" * 70)

# Paramètres
def count_params(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def count_stored_nrc_params(analyzer, selector, nrc_net):
    a1, a2 = analyzer.conv1, analyzer.conv2
    b1, b2 = selector.conv1_bnrc, selector.conv2_bnrc

    stored_conv = (
        b1["BNRC_keep"].size + len(b1["selected_seeds"]) +
        b2["BNRC_keep"].size + len(b2["selected_seeds"])
    )
    stored_dense = (
        sum(p.numel() for p in nrc_net.nrc_conv1.conv_coord.parameters()) +
        sum(p.numel() for p in nrc_net.nrc_conv2.conv_coord.parameters()) +
        sum(p.numel() for layer in [nrc_net.fc1, nrc_net.fc2, nrc_net.fc3]
            for p in layer.parameters())
    )
    stored_meta = a1["w_mean"].size + a2["w_mean"].size
    return stored_conv + stored_dense + stored_meta


total_i, train_i = count_params(initial_net)
total_n, train_n = count_params(nrc_net)
frozen_n         = total_n - train_n
stored_n         = count_stored_nrc_params(analyzer, selector, nrc_net)

# Taille fichiers
size_init = os.path.getsize("initial_weights.pth")
size_nrc  = os.path.getsize("model_nrc.pth")

print(f"\n{'Métrique':<45} {'Initial':>10} {'NRC':>10}")
print("─" * 67)
print(f"  {'Paramètres instanciés totaux':<43} {total_i:>10,} {total_n:>10,}")
print(f"  {'Paramètres entraînables':<43} {train_i:>10,} {train_n:>10,}")
print(f"  {'Paramètres figés en mémoire':<43} {'0':>10} {frozen_n:>10,}")
print(f"  {'Paramètres réellement stockés':<43} {total_i:>10,} {stored_n:>10,}")
print(f"  {'Fichier disque (Ko)':<43} {size_init/1024:>9.1f} {size_nrc/1024:>9.1f}")
print(f"  {'Gain compression fichier':<43} {'—':>10} {(1-size_nrc/size_init)*100:>9.1f}%")

b1, b2 = selector.conv1_bnrc, selector.conv2_bnrc
print(f"\n  conv1 — s PCA (étape 2)      : {analyzer.conv1['s']}")
print(f"  conv1 — n_keep (Vs)          : {b1['n_keep']}")
print(f"  conv1 — seeds LFSR stockées  : {len(b1['selected_seeds'])}")
print(f"  conv1 — rang final t         : {b1['t']}")
print(f"  conv2 — s PCA (étape 2)      : {analyzer.conv2['s']}")
print(f"  conv2 — n_keep (Vs)          : {b2['n_keep']}")
print(f"  conv2 — seeds LFSR stockées  : {len(b2['selected_seeds'])}")
print(f"  conv2 — rang final t         : {b2['t']}")
print(f"\n  conv1 — indices PCA conservés : {analyzer.conv1['important_indices']}")
print(f"  conv1 — directions remplacées : {analyzer.conv1['unimportant_indices']}")
print(f"  conv1 — seeds LFSR            : {b1['selected_seeds']}")
print(f"  conv2 — indices PCA conservés : {analyzer.conv2['important_indices']}")
print(f"  conv2 — directions remplacées : {analyzer.conv2['unimportant_indices']}")
print(f"  conv2 — seeds LFSR            : {b2['selected_seeds']}")

# Visualisation
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("NRC — Modèle Initial vs Compressé", fontsize=14, fontweight="bold")

# Graphe 1 : Taille fichier
labels  = ["Initial", "NRC"]
colors  = ["#4C72B0", "#55A868"]
sizes   = [size_init/1024, size_nrc/1024]
bars    = axes[0].bar(labels, sizes, color=colors, width=0.45)
axes[0].set_ylabel("Taille (Ko)")
axes[0].set_title("Taille fichier sur disque")
for bar, val in zip(bars, sizes):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.5, f"{val:.1f} Ko",
                 ha="center", fontsize=11, fontweight="bold")

# Graphe 2 : Paramètres stockés vs instanciés
x = np.arange(2)
instantiated_vals = [total_i, total_n]
stored_vals       = [total_i, stored_n]
axes[1].bar(x, stored_vals, 0.45, color="#55A868", label="Stockés")
axes[1].bar(x, np.array(instantiated_vals) - np.array(stored_vals), 0.45,
            bottom=stored_vals, color="#DD8452", label="Instanciés seulement", alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels)
axes[1].set_ylabel("Nombre de paramètres")
axes[1].set_title("Paramètres stockés vs instanciés")
axes[1].legend()
for xi, total_val in enumerate(instantiated_vals):
    axes[1].text(xi, total_val + total_val * 0.02, f"{total_val:,}",
                 ha="center", fontsize=9, fontweight="bold")

# Graphe 3 : Répartition des paramètres stockés NRC
labels_pie = ["conv_coord\n(Y_NRC)", "BNRC_keep", "Seeds LFSR", "FC", "w_mean"]
stored_coord = (sum(p.numel() for p in nrc_net.nrc_conv1.conv_coord.parameters()) +
                sum(p.numel() for p in nrc_net.nrc_conv2.conv_coord.parameters()))
stored_bnrc_keep = b1["BNRC_keep"].size + b2["BNRC_keep"].size
stored_seeds = len(b1["selected_seeds"]) + len(b2["selected_seeds"])
stored_fc = sum(p.numel() for ln in [nrc_net.fc1, nrc_net.fc2, nrc_net.fc3]
                for p in ln.parameters())
stored_wmean = analyzer.conv1["w_mean"].size + analyzer.conv2["w_mean"].size
axes[2].pie([stored_coord, stored_bnrc_keep, stored_seeds, stored_fc, stored_wmean],
            labels=labels_pie,
            autopct="%1.1f%%",
            colors=["#55A868", "#4C72B0", "#C44E52", "#8172B3", "#CCB974"],
            startangle=90)
axes[2].set_title("Répartition des données stockées")

plt.tight_layout()
plt.savefig("nrc_summary.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figure sauvegardée → nrc_summary.png")


In [ ]:
# Diagnostic rapide : quelques couples (eTh, p)

def _diag_case(conv, name, eth, p):
    analysis = _analyze_conv_nrc(conv, eth, name)
    bnrc = _build_bnrc(analysis, randomization_ratio=p, K=10, seed=0, name=name)
    print(f"{name:5s} | eTh={eth:0.2f} | p={p:0.2f} | "
          f"s={analysis['s']} | n_keep={bnrc['n_keep']} | "
          f"n_rand={bnrc['n_rand']} | t={bnrc['t']}")

cases = [
    (0.25, 0.00),
    (0.25, 1.00),
    (0.90, 0.00),
    (0.90, 1.00),
]

for eth, p in cases:
    _diag_case(initial_net.conv1, "conv1", eth, p)
for eth, p in cases:
    _diag_case(initial_net.conv2, "conv2", eth, p)
